# CS3264 Assignment 2: Repair the Model

Use this notebook to understand the worked examples and run your own experiments.
Read [the assignment page](https://haroldsoh.com/cs3264/assignments/assignment-02.html) for the assignment requirements, marking rubric, and submission format.
Use `submission_template.json` as the field guide when exporting your own results.

**The two examples are for illustration only. Do not submit them or their numerical
or wording variants as any of your own questions, or use their recorded results
as your experiment evidence.**
You may investigate the same ML topics and adapt the repair ideas.

## 1. Use this notebook

Read the full questions and recorded responses in Section 3 without running code.
To execute the notebook, run the cells in order:

- Keep `MODE="recorded"` to inspect the saved results without downloading a model.
- For live experiments, upload this notebook to [Google Colab](https://colab.research.google.com/),
  select **Runtime → Change runtime type** and pick a GPU (we suggest the T4 when
  Colab offers it; any other NVIDIA GPU, such as the L4, also works), and set
  `INSTALL_INFERENCE_PACKAGES=True`. The model download is approximately 4 GB.
- Set `MODE="live"` only if you also want to rerun the teaching examples in Section 4.
  This adds four model calls. Calls to `run_my_questions`
  in Section 5 always run the model, regardless of `MODE`.

Arrange your own computing access and start early. The teaching team does not
provide GPUs or credits, run experiments for students, or arrange access.
See **Computing resources** on [the assignment page](https://haroldsoh.com/cs3264/assignments/assignment-02.html) for Colab, Kaggle, NUS IT HPC, and
provider support links. Free GPU availability is not guaranteed. Disconnect GPU
runtimes when finished and save your results before leaving a session.

A single long response can take several minutes; runtime depends on your GPU and
the generated response length. Reserve at least 20 calls for a single-call final comparison
and anchor repeats, plus screening and development; multi-call methods need more.
The runner counts saved calls across its working logs, including live examples. Restore all
earlier logs before continuing. Also count interrupted calls with no saved response
and any calls made outside this runner; note these separately in your notebook.

In [1]:
MODE = "recorded"  # Set "live" only to rerun the teaching examples.
INSTALL_INFERENCE_PACKAGES = False
RUN_NAME = "example_run_01"  # Example reruns only; name your own runs in Section 5.
assert MODE in {"recorded", "live"}

## 2. The unmodified baseline

The baseline is **pretrained Qwen3-1.7B** (`Qwen/Qwen3-1.7B`), with its original
weights and **no repair prompt, examples, retrieval, tools, or additional model
calls**. It receives only the fixed system prompt, question text, and final-answer
instruction below. The live baseline uses **the same model and inference settings
as the published Assignment 1 notebook: BF16, Transformers 5.16.1, SDPA attention,
batch size 1, greedy decoding, thinking enabled, and at most 8192 output tokens**.

Keep this baseline unchanged when evaluating your repair. Training may modify
Qwen3-1.7B for the repair arm; it must not modify the comparison baseline.

**System prompt**
```text
You are a careful problem solver. Show your work, then give exactly one final line as instructed.
```
**Instruction appended to every question**
```text
Reason step by step, then end your response with a single line in exactly this format:
Final Answer: <number>
```

In [2]:
#@title Load example data
import copy, html, json, math, platform, re, subprocess, sys, time
from fractions import Fraction
from pathlib import Path
EXAMPLE_QUESTIONS = json.loads("[{\"id\": \"heteroscedastic_mle-00\", \"topic\": \"regression_prob\", \"question\": \"Independent Gaussian observations have a common unknown mean mu. There are 2 observations equal to 1, 1 equal to 7.0, and 3 equal to 3.0. Their known variances, by group, are [1, 4, 9]. What is the maximum-likelihood estimate of mu?\", \"answer\": 1.8387096774193548, \"derivation\": \"Precision and multiplicity weighted mean: sum(n*x/v)/sum(n/v)=1.83870967742.\", \"absolute_tolerance\": 1e-05, \"relative_tolerance\": 0.01, \"split\": \"example\"}, {\"id\": \"bernoulli_absence-01\", \"topic\": \"classification\", \"question\": \"A Bernoulli naive Bayes model has P(Y=1)=0.6. Conditional on Y, the three binary features are independent. P(X_j=1|Y=0), for j=1,2,3, is [0.2, 0.8, 0.3]; P(X_j=1|Y=1) is [0.7, 0.4, 0.6]. Given X1=0, X2=1, X3=0, compute P(Y=1|the observed features). A zero is an observed absence; an unobserved feature, if any, is marginalized.\", \"answer\": 0.13846153846153847, \"derivation\": \"Multiply present/absent feature likelihoods and prior, normalize: 0.028800000000000006/(0.028800000000000006+0.17920000000000003).\", \"absolute_tolerance\": 1e-05, \"relative_tolerance\": 0.01, \"split\": \"example\"}]")
REGRESSION_CANDIDATES = json.loads("[{\"id\": \"shared_weight-00\", \"topic\": \"nn_backprop\", \"question\": \"A scalar network predicts f(w)=w*(w*x+b). Every occurrence of w is the same trainable parameter; x=2, b=1 are fixed. For target y=2, loss L=0.5*(f(w)-y)^2. Starting at w=1, take one gradient descent step with learning rate 0.01. What is the new w?\", \"answer\": 0.95, \"derivation\": \"f=w^2*x+b*w. dL/dw=(f-y)*(2*w^1*x+b)=5; w_new=0.95.\", \"absolute_tolerance\": 1e-05, \"relative_tolerance\": 0.01, \"split\": \"regression\", \"anchor_id\": null, \"variant_type\": null, \"independent_check\": \"Expand L=0.5*(2*w^2+w-2)^2=2*w^4+2*w^3-3.5*w^2-2*w+2. Its derivative at w=1 is 8+6-7-2=5; the update is 1-0.01*5=0.95.\"}, {\"id\": \"shared_weight-01\", \"topic\": \"nn_backprop\", \"question\": \"A scalar network predicts f(w)=w*(w*x+b). Every occurrence of w is the same trainable parameter; x=1, b=2 are fixed. For target y=1, loss L=0.5*(f(w)-y)^2. Starting at w=2, take one gradient descent step with learning rate 0.01. What is the new w?\", \"answer\": 1.58, \"derivation\": \"f=w^2*x+b*w. dL/dw=(f-y)*(2*w^1*x+b)=42; w_new=1.58.\", \"absolute_tolerance\": 1e-05, \"relative_tolerance\": 0.01, \"split\": \"regression\", \"anchor_id\": null, \"variant_type\": null, \"independent_check\": \"Expand L=0.5*(w^2+2*w-1)^2=0.5*w^4+2*w^3+w^2-2*w+0.5. Its derivative at w=2 is 16+24+4-2=42; the update is 2-0.01*42=1.58.\"}, {\"id\": \"simultaneous_update-00\", \"topic\": \"nn_backprop\", \"question\": \"A two-layer scalar linear network has prediction u*v*x and loss L=0.5*(u*v*x-y)^2. Here x=1, y=1, initial u=1, v=2, and learning rate eta=0.025. Perform 1 gradient descent step(s), updating u and v simultaneously at each step using gradients evaluated before that step. What is the loss after the update(s)?\", \"answer\": 0.38390703125, \"derivation\": \"At each step e=u*v*x-y, (u,v)<-(u-eta*e*v*x,v-eta*e*u*x). Final (0.95,1.975); loss=0.38390703125.\", \"absolute_tolerance\": 1e-05, \"relative_tolerance\": 0.01, \"split\": \"regression\", \"anchor_id\": null, \"variant_type\": null, \"independent_check\": \"Holding v=2 gives L(u)=0.5*(2*u-1)^2 and dL/du=2 at u=1. Holding u=1 gives L(v)=0.5*(v-1)^2 and dL/dv=1 at v=2. The simultaneous update is (0.95,1.975); direct substitution gives 0.5*(0.95*1.975-1)^2=0.38390703125.\"}, {\"id\": \"simultaneous_update-01\", \"topic\": \"nn_backprop\", \"question\": \"A two-layer scalar linear network has prediction u*v*x and loss L=0.5*(u*v*x-y)^2. Here x=2, y=1, initial u=2, v=1, and learning rate eta=0.025. Perform 1 gradient descent step(s), updating u and v simultaneously at each step using gradients evaluated before that step. What is the loss after the update(s)?\", \"answer\": 1.2640499999999997, \"derivation\": \"At each step e=u*v*x-y, (u,v)<-(u-eta*e*v*x,v-eta*e*u*x). Final (1.85,0.7); loss=1.2640499999999997.\", \"absolute_tolerance\": 1e-05, \"relative_tolerance\": 0.01, \"split\": \"regression\", \"anchor_id\": null, \"variant_type\": null, \"independent_check\": \"Holding v=1 gives L(u)=0.5*(2*u-1)^2 and dL/du=6 at u=2. Holding u=2 gives L(v)=0.5*(4*v-1)^2 and dL/dv=12 at v=1. The simultaneous update is (1.85,0.7); direct substitution gives 0.5*(2*1.85*0.7-1)^2=1.26405.\"}]")
RECORDED = json.loads("[{\"id\": \"heteroscedastic_mle-00\", \"split\": \"example\", \"method\": \"baseline\", \"question\": \"Independent Gaussian observations have a common unknown mean mu. There are 2 observations equal to 1, 1 equal to 7.0, and 3 equal to 3.0. Their known variances, by group, are [1, 4, 9]. What is the maximum-likelihood estimate of mu?\", \"completion\": \"<think>\\nOkay, let's see. I need to find the maximum-likelihood estimate of mu for these Gaussian observations. The problem says there are 2 observations equal to 1, 1 equal to 7.0, and 3 equal to 3.0. The known variances by group are [1, 4, 9]. \\n\\nFirst, I remember that the maximum-likelihood estimate for the mean of a Gaussian distribution when the variance is known is the sample mean. But wait, here the variances are given by group, so maybe it's a bit different. Let me think.\\n\\nWait, the observations are grouped into different groups with their own variances. So each group has its own variance, and the observations within each group are independent. So the total variance would be a weighted average of the group variances? Or is it something else?\\n\\nNo, actually, the maximum likelihood estimator for the mean when the variance is known is the sample mean. But here, the variances are known, but they are grouped. Wait, the problem says \\\"known variances, by group\\\". So maybe the variances are known for each group, but the overall variance is not known. Wait, but the question is about the maximum-likelihood estimate of mu, which is the mean. \\n\\nWait, the maximum likelihood estimator for the mean when the variance is known is the sample mean. But here, the variances are known for each group. So maybe we need to compute the weighted average of the group means, where each group's weight is the number of observations in that group divided by the total number of observations? Or is it the other way around?\\n\\nWait, no. Let me recall. If you have multiple groups with known variances, and you want to estimate the mean, the maximum likelihood estimator would be the weighted average of the group means, where the weights are the number of observations in each group. Because each observation contributes to the mean. \\n\\nBut wait, the variances are given for each group. But if the variances are known, then the maximum likelihood estimator for the mean is the weighted average of the group means, with weights being the number of observations in each group. \\n\\nSo let's check. Let's say we have three groups. The first group has 2 observations, each with value 1. The second group has 1 observation with value 7.0. The third group has 3 observations with value 3.0. \\n\\nThe group means are 1, 7.0, and 3.0. The number of observations in each group is 2, 1, 3. \\n\\nSo the weighted average would be (2*1 + 1*7 + 3*3) divided by (2 + 1 + 3). Let me compute that. \\n\\n2*1 is 2, 1*7 is 7, 3*3 is 9. Adding those gives 2 + 7 + 9 = 18. The total number of observations is 2 + 1 + 3 = 6. So 18 divided by 6 is 3. \\n\\nWait, but that would be the sample mean. But if the variances are known, then why is the answer 3? Because the sample mean is 3. But let me check again. \\n\\nAlternatively, maybe the variances are given for each group, but the maximum likelihood estimator for the mean is still the overall sample mean. Because the variance is known, but the mean is estimated by the sample mean. \\n\\nWait, but the problem says \\\"known variances, by group\\\". So maybe the variances are known, but the group means are unknown. So the maximum likelihood estimator for the mean would be the weighted average of the group means, where the weights are the number of observations in each group. \\n\\nBut in that case, the group means are 1, 7, 3. So the weighted average is (2*1 + 1*7 + 3*3)/6 = 18/6 = 3. \\n\\nBut that's the same as the overall sample mean. So maybe the answer is 3. \\n\\nWait, but if the variances are known, then the maximum likelihood estimator for the mean is the sample mean. But here, the sample mean is 3. So the answer is 3. \\n\\nBut let me think again. Suppose the variances are known, but the group variances are given. Wait, but the problem says \\\"their known variances, by group are [1, 4, 9]\\\". So the variances for each group are 1, 4, 9. So the group variances are known. \\n\\nBut in the maximum likelihood estimation, if the variances are known, then the estimator for the mean is the sample mean. But if the variances are not known, then we would use the sample variance. But here, the variances are known. \\n\\nWait, but the problem is about the maximum likelihood estimate of mu. So if the variances are known, then the maximum likelihood estimator for mu is the sample mean. \\n\\nBut in this case, the sample mean is (2*1 + 1*7 + 3*3)/6 = 18/6 = 3. So the answer is 3. \\n\\nBut maybe I need to check if the variances are given for each group, but the maximum likelihood estimator for the mean is still the overall sample mean. \\n\\nYes, because the maximum likelihood estimator for the mean is the sample mean regardless of the group variances. Because the group variances are known, but the sample mean is calculated by the sum of the observations divided by the total number of observations. \\n\\nSo the answer is 3. \\n\\nWait, but let me think again. Suppose that the variances are known, but the group variances are different. For example, if you have two groups with different variances, but you know the variances. Then, the maximum likelihood estimator for the mean would still be the overall sample mean. Because the maximum likelihood estimator for the mean is the same regardless of the variances. \\n\\nYes, because the likelihood function for the mean is the product of the normal distributions for each group, and the maximum likelihood estimator is the value that maximizes this product. \\n\\nSo, the maximum likelihood estimator for the mean is the weighted average of the group means, where the weights are the number of observations in each group. But in this case, the group means are 1, 7, 3. So the weighted average is 3. \\n\\nTherefore, the maximum-likelihood estimate of mu is 3. \\n\\nFinal Answer: 3\\n</think>\\n\\nFinal Answer: 3\", \"status\": \"incorrect\", \"predicted\": 3.0, \"truncated\": false, \"input_tokens\": 124, \"output_tokens\": 1396, \"seconds\": 86.28848380001727, \"peak_allocated_gib\": 3.4232497215270996, \"origin\": \"recorded_example\", \"started_at_unix\": 1791787485.258758, \"finished_at_unix\": 1791787571.5780506}, {\"id\": \"bernoulli_absence-01\", \"split\": \"example\", \"method\": \"baseline\", \"question\": \"A Bernoulli naive Bayes model has P(Y=1)=0.6. Conditional on Y, the three binary features are independent. P(X_j=1|Y=0), for j=1,2,3, is [0.2, 0.8, 0.3]; P(X_j=1|Y=1) is [0.7, 0.4, 0.6]. Given X1=0, X2=1, X3=0, compute P(Y=1|the observed features). A zero is an observed absence; an unobserved feature, if any, is marginalized.\", \"completion\": \"<think>\\nOkay, let's try to figure out this problem step by step. So, we have a Bernoulli Naive Bayes model, and we need to compute the probability P(Y=1 | X1=0, X2=1, X3=0). \\n\\nFirst, I remember that Naive Bayes works by applying Bayes' theorem. The formula is P(Y=1 | X1, X2, X3) = [P(X1, X2, X3 | Y=1) * P(Y=1)] / P(X1, X2, X3). \\n\\nSince the features are binary and independent given Y, we can compute P(X1, X2, X3 | Y=1) as the product of individual probabilities. Similarly for Y=0. \\n\\nGiven that the features are independent, P(X1, X2, X3 | Y=1) = P(X1=0 | Y=1) * P(X2=1 | Y=1) * P(X3=0 | Y=1). \\n\\nLooking at the problem, P(X_j=1 | Y=1) for j=1,2,3 is [0.7, 0.4, 0.6]. So for X1=0, which is the first feature, P(X1=0 | Y=1) is 1 - 0.7 = 0.3. Similarly, X2=1 is 0.4, and X3=0 is 1 - 0.6 = 0.4. \\n\\nSo multiplying those together: 0.3 * 0.4 * 0.4. Let me calculate that. 0.3 * 0.4 is 0.12, then 0.12 * 0.4 is 0.048. So P(X1, X2, X3 | Y=1) is 0.048.\\n\\nNow, for Y=0, the probabilities are P(X_j=1 | Y=0) = [0.2, 0.8, 0.3]. So for X1=0, which is the first feature, P(X1=0 | Y=0) is 1 - 0.2 = 0.8. X2=1 is 0.8, and X3=0 is 0.3. So multiplying those: 0.8 * 0.8 * 0.3. Let's compute that. 0.8 * 0.8 is 0.64, then 0.64 * 0.3 is 0.192. So P(X1, X2, X3 | Y=0) is 0.192.\\n\\nNext, we need P(X1, X2, X3). Since the features are independent, this is the product of their individual probabilities. Wait, but actually, since we're using the formula P(Y=1 | X1, X2, X3) = [P(X1, X2, X3 | Y=1) * P(Y=1)] / P(X1, X2, X3), and P(X1, X2, X3) is the total probability of the features, which can be calculated as P(X1, X2, X3 | Y=1) * P(Y=1) + P(X1, X2, X3 | Y=0) * P(Y=0). \\n\\nBut wait, actually, the denominator is P(X1, X2, X3), which is the sum over Y=0 and Y=1 of P(Y) * P(X1, X2, X3 | Y). So we need to compute that. \\n\\nSo let's compute the numerator first. The numerator is P(Y=1) * P(X1, X2, X3 | Y=1) = 0.6 * 0.048 = 0.0288. \\n\\nThen the denominator is P(X1, X2, X3) = P(Y=0) * P(X1, X2, X3 | Y=0) + P(Y=1) * P(X1, X2, X3 | Y=1) = 0.4 * 0.192 + 0.6 * 0.048. Let's calculate that. \\n\\n0.4 * 0.192 is 0.0768, and 0.6 * 0.048 is 0.0288. Adding them together: 0.0768 + 0.0288 = 0.1056. \\n\\nSo the final probability is 0.0288 / 0.1056. Let me compute that. Dividing numerator and denominator by 0.0288 gives 1 / 3.6, which is approximately 0.2778. Wait, but let me check the exact calculation. \\n\\n0.0288 divided by 0.1056. Let's see, 0.0288 / 0.1056 = (0.0288 / 0.0288) / (0.1056 / 0.0288) = 1 / 3.666... which is approximately 0.2727. Wait, maybe I made a mistake in the division. Let me do it step by step. \\n\\n0.0288 divided by 0.1056. Multiply numerator and denominator by 1000 to eliminate decimals: 28.8 / 105.6. Then divide numerator and denominator by 28.8: 1 / (105.6 / 28.8) = 1 / 3.666... which is 1 / (11/3) = 3/11 \\u2248 0.2727. \\n\\nSo the final answer is approximately 0.2727. But let me check if I did all steps correctly. \\n\\nWait, let me recheck the calculations. \\n\\nFirst, P(X1, X2, X3 | Y=1) is 0.3 * 0.4 * 0.4 = 0.048. Correct. \\n\\nP(Y=1) is 0.6. So numerator is 0.6 * 0.048 = 0.0288. \\n\\nFor the denominator, P(X1, X2, X3) = 0.4 * 0.192 + 0.6 * 0.048. \\n\\n0.4 * 0.192: 0.4 * 0.192 = 0.0768. \\n\\n0.6 * 0.048 = 0.0288. \\n\\nAdding them gives 0.0768 + 0.0288 = 0.1056. \\n\\nSo 0.0288 / 0.1056 = 0.2727... which is 3/11. \\n\\nYes, because 3 divided by 11 is approximately 0.2727. \\n\\nSo the final answer is 3/11, which is approximately 0.2727. But the problem says to compute it, so maybe they want the exact fraction or a decimal? The question says to put the final answer as a number, so probably as a decimal rounded to a certain place, but since the original probabilities are given with two decimal places, maybe we can present it as 0.2727 or 0.273. But let me check if there's a calculation mistake. \\n\\nAlternatively, maybe I made a mistake in calculating P(X1, X2, X3 | Y=0). Let me check that again. \\n\\nFor Y=0, P(X1=0 | Y=0) is 1 - 0.2 = 0.8. X2=1 is 0.8, X3=0 is 0.3. So 0.8 * 0.8 * 0.3 = 0.192. Correct. \\n\\nThen P(Y=0) is 0.4, so 0.4 * 0.192 = 0.0768. \\n\\nAnd for Y=1, 0.6 * 0.048 = 0.0288. \\n\\nTotal denominator is 0.0768 + 0.0288 = 0.1056. \\n\\nYes, that's correct. \\n\\nSo 0.0288 / 0.1056 = 0.2727... which is 3/11. \\n\\nSo the exact value is 3/11, which is approximately 0.2727. Depending on the required precision, but since the problem gives probabilities with two decimal places, maybe we need to round to three decimal places. But the question says \\\"compute P(Y=1|the observed features)\\\", so maybe they want the exact value. However, in practice, it's often presented as a decimal. \\n\\nAlternatively, maybe there's a calculation mistake. Let me check once more. \\n\\nAlternatively, maybe I should use fractions instead of decimals. Let's try that. \\n\\nP(X1, X2, X3 | Y=1) = 0.3 * 0.4 * 0.4 = (3/10) * (2/5) * (4/10) = (3*2*4)/(10*5*10) = 24/500 = 6/125. \\n\\nP(Y=1) is 3/5. So numerator is 3/5 * 6/125 = 18/625. \\n\\nDenominator: P(X1, X2, X3) = P(Y=0) * P(X1, X2, X3 | Y=0) + P(Y=1) * P(X1, X2, X3 | Y=1) \\n\\nP(Y=0) is 2/5, P(X1, X2, X3 | Y=0) is 0.8 * 0.8 * 0.3 = (4/5) * (4/5) * (3/10) = (16/25) * (3/10) = 48/250 = 24/125. \\n\\nSo P(Y=0) * P(X1, X2, X3 | Y=0) = 2/5 * 24/125 = 48/625. \\n\\nP(Y=1) * P(X1, X2, X3 | Y=1) = 3/5 * 6/125 = 18/625. \\n\\nTotal denominator is 48/625 + 18/625 = 66/625. \\n\\nSo the probability is (18/625) / (66/625) = 18/66 = 3/11 \\u2248 0.2727. \\n\\nYes, so the exact value is 3/11, which is approximately 0.2727. \\n\\nTherefore, the final answer is 0.2727, but maybe they want it as a fraction? But the problem says to put the final answer as a number, so probably decimal. But maybe they want it as a fraction. However, the initial probabilities are given as decimals, so maybe the answer is expected as a decimal. \\n\\nBut let me check if there's any other step I missed. For example, the problem says \\\"a zero is an observed absence; an unobserved feature, if any, is marginalized.\\\" Wait, does that affect anything here? \\n\\nWait, the features are given as X1=0, X2=1, X3=0. So all features are observed. So the denominator is P(X1, X2, X3) which is the sum over Y=0 and Y=1 of P(Y) * P(X1, X2, X3 | Y). Which we already calculated. \\n\\nSo I think the steps are correct. Therefore, the final answer is 3/11, which is approximately 0.2727. Depending on the required precision, but since the problem didn't specify, I think 0.2727 is acceptable. However, maybe the exact fraction is better. But the question says to put the final answer as a number, so probably decimal. \\n\\nAlternatively, maybe the answer is 0.2727, but let me check with exact decimal. 3 divided by 11 is 0.272727..., so 0.2727 when rounded to four decimal places. \\n\\nBut the problem might expect a fraction. However, the initial probabilities are given as decimals, so maybe the answer is expected as a decimal. \\n\\nSo I think the final answer is 0.2727, but let me check once more. \\n\\nYes, 3/11 is approximately 0.2727. So the answer is 0.2727. But maybe the problem expects rounding to three decimal places, so 0.273. But since the exact value is 0.2727..., it's better to present it as 0.2727. However, in some contexts, fractions are preferred. But the problem says \\\"compute P(Y=1|the observed features)\\\", and the answer is a number. \\n\\nAlternatively, maybe I made a mistake in the calculation. Let me check once again. \\n\\nNumerator: 0.6 * 0.048 = 0.0288. \\n\\nDenominator: 0.4 * 0.192 + 0.6 * 0.048 = 0.0768 + 0.0288 = 0.1056. \\n\\n0.0288 / 0.1056 = 0.2727... \\n\\nYes, that's correct. So the answer is 0.2727. \\n\\nBut maybe the problem expects it as a fraction. However, the user said to put the final answer as a number, so I think decimal is okay. \\n\\nFinal Answer: 0.2727\\n</think>\\n\\nFinal Answer: 0.2727\", \"status\": \"incorrect\", \"predicted\": 0.2727, \"truncated\": false, \"input_tokens\": 192, \"output_tokens\": 3187, \"seconds\": 191.1283690999844, \"peak_allocated_gib\": 3.6834020614624023, \"origin\": \"recorded_example\", \"started_at_unix\": 1791787571.579058, \"finished_at_unix\": 1791787762.7297926}, {\"id\": \"heteroscedastic_mle-00\", \"split\": \"example\", \"method\": \"likelihood_card\", \"question\": \"Independent Gaussian observations have a common unknown mean mu. There are 2 observations equal to 1, 1 equal to 7.0, and 3 equal to 3.0. Their known variances, by group, are [1, 4, 9]. What is the maximum-likelihood estimate of mu?\", \"completion\": \"<think>\\nOkay, let's see. I need to find the maximum-likelihood estimate of mu for these Gaussian observations. The problem says there are 2 observations equal to 1, 1 equal to 7.0, and 3 equal to 3.0. The known variances by group are [1, 4, 9]. So each group has different variances. \\n\\nFirst, I remember that the maximum-likelihood estimate for the mean of a Gaussian distribution with known group variances is given by the formula: mu = [sum_g (n_g * x_g / v_g)] / [sum_g (n_g / v_g)]. So I need to compute the numerator and denominator separately.\\n\\nLet me break down the data. There are three groups. The first group has 2 observations of 1, with variance 1. The second group has 1 observation of 7.0, variance 4. The third group has 3 observations of 3.0, variance 9.\\n\\nFor the numerator, I need to calculate for each group the (n_g * x_g) divided by v_g, then sum them all. Let's do that group by group.\\n\\nFirst group: n_g = 2, x_g = 1, v_g = 1. So (2 * 1)/1 = 2. \\n\\nSecond group: n_g = 1, x_g = 7.0, v_g = 4. So (1 * 7.0)/4 = 7.0 / 4 = 1.75.\\n\\nThird group: n_g = 3, x_g = 3.0, v_g = 9. So (3 * 3.0)/9 = 9 / 9 = 1. \\n\\nAdding these up: 2 + 1.75 + 1 = 4.75. So the numerator is 4.75.\\n\\nNow the denominator is sum_g (n_g / v_g). Let's compute each term.\\n\\nFirst group: 2 / 1 = 2.\\n\\nSecond group: 1 / 4 = 0.25.\\n\\nThird group: 3 / 9 = 0.333... (which is 1/3).\\n\\nAdding them: 2 + 0.25 + 0.333... = 2.58333...\\n\\nSo the denominator is approximately 2.58333.\\n\\nThen the maximum-likelihood estimate mu is numerator divided by denominator: 4.75 / 2.58333.\\n\\nLet me calculate that. Let's see, 4.75 divided by 2.58333. Let me convert that to fractions to be precise. \\n\\nThe numerator is 4.75, which is 19/4. The denominator is 2 + 1/4 + 1/3. Let's convert that to a common denominator. The denominators are 1, 4, 3. The least common denominator is 12. So 2 is 24/12, 1/4 is 3/12, 1/3 is 4/12. Adding them: 24 + 3 + 4 = 31/12. So denominator is 31/12. \\n\\nSo mu = (19/4) / (31/12) = (19/4) * (12/31) = (19 * 12) / (4 * 31) = (19 * 3) / 31 = 57/31. \\n\\nCalculating 57 divided by 31. Let me do that division. 31*1=31, 57-31=26. So 1 + 26/31 \\u2248 1.8387. \\n\\nBut wait, let me check the calculation again. 19/4 is 4.75, and 31/12 is approximately 2.5833. So 4.75 divided by 2.5833. Let me compute 4.75 / 2.5833. Let me multiply numerator and denominator by 1000 to eliminate decimals: 4750 / 2583.3. Hmm, maybe better to do decimal division. \\n\\nAlternatively, since 57/31 is approximately 1.8387. Let me check if that's correct. 31*1.8387 \\u2248 31*1.8 + 31*0.0387 \\u2248 55.8 + 1.20 = 57.0. Yes, that's correct. So the exact value is 57/31, which is approximately 1.8387. \\n\\nBut the question says to put the final answer as a number. So maybe they want the exact fraction or the decimal? The problem says \\\"maximum-likelihood estimate of mu\\\" and doesn't specify, but since the numbers are given as integers and variances as integers, maybe the answer is a fraction. But 57/31 is the exact value. However, maybe I made a mistake in the calculation. Let me check again.\\n\\nWait, the numerator was 2 + 1.75 + 1 = 4.75. The denominator was 2 + 0.25 + 0.333... = 2.58333... which is 31/12. So 4.75 divided by 31/12 is 4.75 * 12/31. 4.75 * 12 = 57. So 57/31. Yes, that's correct. So the exact value is 57/31, which is approximately 1.8387. \\n\\nBut maybe the problem expects a decimal? Let me check if 57 divided by 31 is indeed the correct fraction. 31*1.8 = 55.8, 57 - 55.8 = 1.2. 1.2/31 \\u2248 0.0387. So yes, 1.8387. \\n\\nBut the problem says \\\"use these mathematical reference notes only when relevant.\\\" So I think the answer is 57/31, but maybe they want a decimal. However, since the original data has integers, perhaps the answer is a fraction. But the question says \\\"maximum-likelihood estimate of mu\\\" and in statistics, it's common to present it as a decimal. But maybe they want the exact fraction. However, the problem might have a simpler answer. Let me check my steps again.\\n\\nWait, the group variances are [1,4,9]. The group sizes are [2,1,3]. So for each group, the contribution to the numerator is (n_g * x_g)/v_g. So for the first group: 2*1/1 = 2. Second group: 1*7/4 = 7/4 = 1.75. Third group: 3*3/9 = 1. So total numerator is 2 + 1.75 + 1 = 4.75. Denominator is 2/1 + 1/4 + 3/9 = 2 + 0.25 + 0.333... = 2.58333... which is 31/12. So 4.75 divided by 31/12 is 4.75 * 12/31 = 57/31. So yes, that's correct. \\n\\nTherefore, the maximum-likelihood estimate is 57/31, which is approximately 1.8387. But since the problem might expect a decimal, maybe they want it as a decimal. However, the exact value is 57/31. But let me check if there's a calculation mistake. \\n\\nAlternatively, maybe I miscalculated the denominator. Let me recheck the denominator. \\n\\nFirst group: 2 / 1 = 2. Second group: 1 / 4 = 0.25. Third group: 3 / 9 = 0.333... So 2 + 0.25 + 0.333... = 2.58333... which is 2 + 1/4 + 1/3. Let me compute 1/4 + 1/3 = 3/12 + 4/12 = 7/12. So total denominator is 2 + 7/12 = 31/12. Yes, that's correct. \\n\\nSo 4.75 is 19/4. So 19/4 divided by 31/12 is (19/4)*(12/31) = (19*12)/(4*31) = (19*3)/31 = 57/31. Correct. \\n\\nSo the final answer is 57/31, which is approximately 1.8387. But the problem might want the exact fraction. However, in some contexts, decimal is preferred. But since the problem didn't specify, but the initial data is in integers, maybe the answer is 57/31. But let me check if 57/31 can be simplified. 57 and 31 have no common factors except 1, since 31 is a prime number. So 57/31 is the simplest form. \\n\\nBut maybe the problem expects a decimal. Let me compute 57 divided by 31. 31*1.8 = 55.8, 57 - 55.8 = 1.2. 1.2/31 = 0.0387. So 1.8 + 0.0387 = 1.8387. So approximately 1.8387. \\n\\nBut the problem says \\\"put your final answer within \\\\boxed{}\\\". In math problems, fractions are often preferred unless specified otherwise. However, sometimes decimal is acceptable. But since the exact value is 57/31, which is a precise fraction, I should present that. But maybe the problem expects a decimal. Let me check if there's a calculation mistake. \\n\\nAlternatively, maybe I made a mistake in the numerator. Let me check again. \\n\\nFirst group: 2 observations of 1. So (2*1)/1 = 2. Correct. \\n\\nSecond group: 1 observation of 7.0. (1*7.0)/4 = 1.75. Correct. \\n\\nThird group: 3 observations of 3.0. (3*3.0)/9 = 1. Correct. \\n\\nTotal numerator: 2 + 1.75 + 1 = 4.75. Correct. \\n\\nDenominator: 2/1 + 1/4 + 3/9 = 2 + 0.25 + 0.333... = 2.58333... Correct. \\n\\nSo 4.75 / 2.58333... = 57/31 \\u2248 1.8387. \\n\\nTherefore, the final answer is 57/31, which is approximately 1.8387. But since the problem might want a decimal, but the exact value is 57/31. However, in the context of maximum likelihood estimation, it's common to present it as a decimal. But since the user hasn't specified, but the initial data is in integers, maybe the answer is 57/31. However, I need to check if there's a simpler way. \\n\\nAlternatively, maybe I made a mistake in the formula. Wait, the formula is mu = [sum (n_g x_g / v_g)] / [sum (n_g / v_g)]. Yes, that's correct. So the calculation is correct. \\n\\nTherefore, the final answer is 57/31, which is approximately 1.8387. But since the problem might want a decimal, I'll go with the exact fraction. However, the user might expect a decimal. Let me check if 57 divided by 31 is indeed the correct value. \\n\\nYes, 31*1.8387 \\u2248 57. So the exact value is 57/31. Therefore, the final answer is 57/31. But the problem says \\\"put your final answer within \\\\boxed{}\\\". In many cases, fractions are preferred. So I'll go with 57/31. But maybe the problem expects a decimal. However, since the user hasn't specified, but the initial data is in integers, I think the exact fraction is better. \\n\\nBut let me check if 57/31 can be simplified. 57 divided by 3 is 19, 31 is prime. So no, it's reduced. So the answer is 57/31. \\n\\nFinal Answer: \\\\boxed{\\\\dfrac{57}{31}}\\n</think>\\n\\nFinal Answer: \\\\boxed{\\\\dfrac{57}{31}}\", \"status\": \"correct\", \"predicted\": 1.8387096774193548, \"truncated\": false, \"input_tokens\": 350, \"output_tokens\": 2904, \"seconds\": 224.10023540002294, \"peak_allocated_gib\": 3.6726765632629395, \"origin\": \"recorded_example\", \"started_at_unix\": 1791788057.0415154, \"finished_at_unix\": 1791788281.2237906}, {\"id\": \"bernoulli_absence-01\", \"split\": \"example\", \"method\": \"likelihood_card\", \"question\": \"A Bernoulli naive Bayes model has P(Y=1)=0.6. Conditional on Y, the three binary features are independent. P(X_j=1|Y=0), for j=1,2,3, is [0.2, 0.8, 0.3]; P(X_j=1|Y=1) is [0.7, 0.4, 0.6]. Given X1=0, X2=1, X3=0, compute P(Y=1|the observed features). A zero is an observed absence; an unobserved feature, if any, is marginalized.\", \"completion\": \"<think>\\nOkay, let's try to figure out this problem step by step. So, we have a Bernoulli Naive Bayes model, and we need to compute P(Y=1 | X1=0, X2=1, X3=0). \\n\\nFirst, let me recall the formula for Bernoulli Naive Bayes. The probability of Y=1 given the features is calculated by taking the prior probability of Y=1, multiplied by the product of the probabilities of each feature given Y=1, and then divided by the same for Y=0. But since the features are binary, we need to compute the likelihoods for both classes.\\n\\nThe prior P(Y=1) is given as 0.6. So that's straightforward. Now, for Y=1, the probabilities of the features are [0.7, 0.4, 0.6]. For Y=0, the probabilities are [0.2, 0.8, 0.3]. \\n\\nThe observed features are X1=0, X2=1, X3=0. So, for Y=1, we need to check if each feature is 1 or 0. Since X1 is 0, that means for Y=1, the probability of X1=0 is (1 - P(X1=1|Y=1)) = 1 - 0.7 = 0.3. Similarly, X2=1 is 0.4, and X3=0 is 1 - 0.6 = 0.4. \\n\\nWait, but the problem says that the features are independent. So for each feature, the probability of being 1 given Y is given, and if it's 0, it's 1 - that probability. So for Y=1, the likelihood is product of P(X1=0|Y=1) * P(X2=1|Y=1) * P(X3=0|Y=1). Which would be 0.3 * 0.4 * 0.4. Let me calculate that: 0.3 * 0.4 is 0.12, then 0.12 * 0.4 is 0.048.\\n\\nFor Y=0, the likelihood is product of P(X1=0|Y=0) * P(X2=1|Y=0) * P(X3=0|Y=0). Since X1=0, that's 0.2. X2=1 is 0.8, and X3=0 is 0.3. So 0.2 * 0.8 * 0.3. Let's compute that: 0.2 * 0.8 is 0.16, then 0.16 * 0.3 is 0.048.\\n\\nSo the numerator for Y=1 is prior * likelihood for Y=1, which is 0.6 * 0.048. Let me calculate that: 0.6 * 0.048 = 0.0288. \\n\\nFor Y=0, the numerator is 0.4 * 0.048. Wait, no. Wait, the prior is 0.4 for Y=0, so the numerator is 0.4 * 0.048. Let me check: 0.4 * 0.048 = 0.0192. \\n\\nThen, the total probability is (0.0288) / (0.0192 + 0.0288) = 0.0288 / 0.048 = 0.6. Wait, that can't be right. Because if the prior is 0.6, and the likelihoods are 0.048 for each class, then the total would be 0.6 * 0.048 / (0.4 * 0.048 + 0.6 * 0.048) = 0.6 / (1) = 0.6. But that would mean P(Y=1 | features) is 0.6, which is the same as the prior. But that seems odd. \\n\\nWait, maybe I made a mistake here. Let me check again. \\n\\nWait, the prior is 0.6 for Y=1. The likelihood for Y=1 is 0.048, so the numerator for Y=1 is 0.6 * 0.048 = 0.0288. For Y=0, the prior is 0.4, and the likelihood is 0.048, so the numerator is 0.4 * 0.048 = 0.0192. Then the total is 0.0288 + 0.0192 = 0.048. So the probability is 0.0288 / 0.048 = 0.6. \\n\\nBut that's the same as the prior. That seems counterintuitive. But maybe that's correct because the features are not providing any information here. Wait, but the features are X1=0, X2=1, X3=0. For Y=1, the features are 0,1,0. For Y=0, the features would be 0,1,0 as well? Wait, no. Wait, the features are given as X1=0, X2=1, X3=0. So for Y=0, the features are 0,1,0. But the probabilities for Y=0 are [0.2, 0.8, 0.3]. So the likelihood for Y=0 is 0.2 * 0.8 * 0.3 = 0.048. \\n\\nBut if the features are the same for both classes, then the likelihoods are the same, so the ratio would be prior / (prior + 0.4 * 0.048 / 0.4 * 0.048 + 0.6 * 0.048 / 0.4 * 0.048)? Wait, no. Wait, the total probability is (prior_Y1 * likelihood_Y1) / (prior_Y0 * likelihood_Y0 + prior_Y1 * likelihood_Y1). \\n\\nWait, no. The formula is P(Y=1 | X) = [P(Y=1) * P(X | Y=1)] / [P(Y=1) * P(X | Y=1) + P(Y=0) * P(X | Y=0)]. \\n\\nSo in this case, P(Y=1) is 0.6, P(X | Y=1) is 0.048, P(Y=0) is 0.4, P(X | Y=0) is 0.048. So the numerator is 0.6 * 0.048 = 0.0288, denominator is 0.4 * 0.048 + 0.6 * 0.048 = 0.048. So 0.0288 / 0.048 = 0.6. \\n\\nSo the answer is 0.6. But that seems to suggest that the features don't affect the probability, which is possible if the features are not informative. But let me check again. \\n\\nWait, maybe I made a mistake in calculating the likelihoods. Let me recheck. \\n\\nFor Y=1, the features are X1=0, X2=1, X3=0. So for each feature, the probability of being 1 given Y=1 is 0.7, 0.4, 0.6. So for X1=0, it's 1 - 0.7 = 0.3. X2=1 is 0.4. X3=0 is 1 - 0.6 = 0.4. So the product is 0.3 * 0.4 * 0.4 = 0.048. Correct. \\n\\nFor Y=0, the features are X1=0, X2=1, X3=0. So for Y=0, the probabilities are 0.2, 0.8, 0.3. So X1=0 is 0.2, X2=1 is 0.8, X3=0 is 0.3. Product is 0.2 * 0.8 * 0.3 = 0.048. Correct. \\n\\nSo the numerator for Y=1 is 0.6 * 0.048 = 0.0288. Denominator is 0.4 * 0.048 + 0.6 * 0.048 = 0.048. So 0.0288 / 0.048 = 0.6. \\n\\nSo the final answer is 0.6. But that seems to suggest that the features don't affect the probability, which is possible if the features are independent and the prior is 0.6. But maybe that's correct. \\n\\nAlternatively, maybe I need to consider that the features are not independent? Wait, the problem says the features are independent. So the calculation is correct. \\n\\nTherefore, the answer is 0.6.\\n</think>\\n\\nFinal Answer: 0.6\", \"status\": \"incorrect\", \"predicted\": 0.6, \"truncated\": false, \"input_tokens\": 418, \"output_tokens\": 2054, \"seconds\": 146.74571980000474, \"peak_allocated_gib\": 3.5602025985717773, \"origin\": \"recorded_example\", \"started_at_unix\": 1791788281.2257924, \"finished_at_unix\": 1791788428.0081694}]")
RECORDED_SETTINGS = json.loads("{\"model\": \"Qwen/Qwen3-1.7B\", \"dtype\": \"bfloat16\", \"batch_size\": 1, \"attn_implementation\": \"sdpa\", \"thinking\": true, \"do_sample\": false, \"max_new_tokens\": 8192, \"seed\": 3264, \"device\": \"NVIDIA GeForce RTX 4060\", \"torch\": \"2.12.0+cu126\", \"transformers\": \"5.16.1\", \"accelerate\": \"1.12.0\", \"platform\": \"Windows-10-10.0.26200-SP0\", \"python\": \"3.11.15\", \"versions\": {\"torch\": \"2.12.0+cu126\", \"transformers\": \"5.16.1\", \"accelerate\": \"1.12.0\"}, \"system_prompt\": \"You are a careful problem solver. Show your work, then give exactly one final line as instructed.\", \"suffix\": \"\\n\\nReason step by step, then end your response with a single line in exactly this format:\\nFinal Answer: <number>\", \"prompts\": {\"baseline\": \"\", \"generic_check\": \"Before giving the final answer, check your reasoning and arithmetic carefully and correct any mistakes you find.\", \"likelihood_card\": \"Use these mathematical reference notes only when relevant. Translate the stated observations into likelihood factors before substituting numbers. For independent Gaussian observations with common mean mu and known group variances v_g, with n_g observations equal to x_g, the stationary equation is mu * sum_g(n_g / v_g) = sum_g(n_g * x_g / v_g). Compute the numerator sum_g(n_g * x_g / v_g) and denominator sum_g(n_g / v_g) separately, then divide. Group-dependent variances cannot be canceled across a sum. Use the ordinary mean only when the variances are equal. If standard deviations are given, square them first. For Bernoulli naive Bayes, compute each class score as prior_c * product_j[p_cj^x_j * (1-p_cj)^(1-x_j)], then normalize the class scores. For every observed zero, use 1-p_cj separately in BOTH classes; only an unobserved feature is marginalized. Show a compact calculation of the intermediate sums or class scores, check the requested quantity, and do not repeat completed derivations.\"}}")
PROMPTS = copy.deepcopy(RECORDED_SETTINGS["prompts"])
assert PROMPTS["baseline"] == ""
print("Loaded two illustrative examples and four regression-check candidates.")

Loaded two illustrative examples and four regression-check candidates.


## 3. Example: from failure to repair

Both original responses below are completed **Qwen3-1.7B baseline** runs, with no
repair added. The baseline and repaired responses use the A1-aligned settings in
Section 2, recorded on an **NVIDIA GeForce RTX 4060**. Responses can differ across
runtimes; inspect the results from your own setup. The expandable panels contain
the complete recorded text, including thinking and the final answer.

### Example A: Gaussian mean with unequal known variances

**Question**

Independent Gaussian observations have a common unknown mean mu. There are 2 observations equal to 1, 1 equal to 7.0, and 3 equal to 3.0. Their known variances, by group, are [1, 4, 9]. What is the maximum-likelihood estimate of mu?

**Correct solution.** The log-likelihood is
$$
\ell(\mu)=C-\frac12\left[2(1-\mu)^2+\frac{(7-\mu)^2}{4}+\frac{3(3-\mu)^2}{9}\right].
$$
Setting its derivative to zero gives
$$
\hat\mu=\frac{2(1)/1+1(7)/4+3(3)/9}{2/1+1/4+3/9}
=\frac{57}{31}\approx1.83871.
$$
The second derivative is $-31/12<0$, so this is the unique maximum.

**Baseline answer: 3 — incorrect.** The model computes the ordinary average
correctly but chooses the wrong estimator. Unequal known variances require
inverse-variance weighting.

<details>
<summary><strong>Original Qwen3-1.7B answer: 3 — expand full response</strong></summary>
<pre style="white-space:pre-wrap;overflow-wrap:anywhere;max-height:none;">&lt;think&gt;
Okay, let&#x27;s see. I need to find the maximum-likelihood estimate of mu for these Gaussian observations. The problem says there are 2 observations equal to 1, 1 equal to 7.0, and 3 equal to 3.0. The known variances by group are [1, 4, 9]. 

First, I remember that the maximum-likelihood estimate for the mean of a Gaussian distribution when the variance is known is the sample mean. But wait, here the variances are given by group, so maybe it&#x27;s a bit different. Let me think.

Wait, the observations are grouped into different groups with their own variances. So each group has its own variance, and the observations within each group are independent. So the total variance would be a weighted average of the group variances? Or is it something else?

No, actually, the maximum likelihood estimator for the mean when the variance is known is the sample mean. But here, the variances are known, but they are grouped. Wait, the problem says &quot;known variances, by group&quot;. So maybe the variances are known for each group, but the overall variance is not known. Wait, but the question is about the maximum-likelihood estimate of mu, which is the mean. 

Wait, the maximum likelihood estimator for the mean when the variance is known is the sample mean. But here, the variances are known for each group. So maybe we need to compute the weighted average of the group means, where each group&#x27;s weight is the number of observations in that group divided by the total number of observations? Or is it the other way around?

Wait, no. Let me recall. If you have multiple groups with known variances, and you want to estimate the mean, the maximum likelihood estimator would be the weighted average of the group means, where the weights are the number of observations in each group. Because each observation contributes to the mean. 

But wait, the variances are given for each group. But if the variances are known, then the maximum likelihood estimator for the mean is the weighted average of the group means, with weights being the number of observations in each group. 

So let&#x27;s check. Let&#x27;s say we have three groups. The first group has 2 observations, each with value 1. The second group has 1 observation with value 7.0. The third group has 3 observations with value 3.0. 

The group means are 1, 7.0, and 3.0. The number of observations in each group is 2, 1, 3. 

So the weighted average would be (2*1 + 1*7 + 3*3) divided by (2 + 1 + 3). Let me compute that. 

2*1 is 2, 1*7 is 7, 3*3 is 9. Adding those gives 2 + 7 + 9 = 18. The total number of observations is 2 + 1 + 3 = 6. So 18 divided by 6 is 3. 

Wait, but that would be the sample mean. But if the variances are known, then why is the answer 3? Because the sample mean is 3. But let me check again. 

Alternatively, maybe the variances are given for each group, but the maximum likelihood estimator for the mean is still the overall sample mean. Because the variance is known, but the mean is estimated by the sample mean. 

Wait, but the problem says &quot;known variances, by group&quot;. So maybe the variances are known, but the group means are unknown. So the maximum likelihood estimator for the mean would be the weighted average of the group means, where the weights are the number of observations in each group. 

But in that case, the group means are 1, 7, 3. So the weighted average is (2*1 + 1*7 + 3*3)/6 = 18/6 = 3. 

But that&#x27;s the same as the overall sample mean. So maybe the answer is 3. 

Wait, but if the variances are known, then the maximum likelihood estimator for the mean is the sample mean. But here, the sample mean is 3. So the answer is 3. 

But let me think again. Suppose the variances are known, but the group variances are given. Wait, but the problem says &quot;their known variances, by group are [1, 4, 9]&quot;. So the variances for each group are 1, 4, 9. So the group variances are known. 

But in the maximum likelihood estimation, if the variances are known, then the estimator for the mean is the sample mean. But if the variances are not known, then we would use the sample variance. But here, the variances are known. 

Wait, but the problem is about the maximum likelihood estimate of mu. So if the variances are known, then the maximum likelihood estimator for mu is the sample mean. 

But in this case, the sample mean is (2*1 + 1*7 + 3*3)/6 = 18/6 = 3. So the answer is 3. 

But maybe I need to check if the variances are given for each group, but the maximum likelihood estimator for the mean is still the overall sample mean. 

Yes, because the maximum likelihood estimator for the mean is the sample mean regardless of the group variances. Because the group variances are known, but the sample mean is calculated by the sum of the observations divided by the total number of observations. 

So the answer is 3. 

Wait, but let me think again. Suppose that the variances are known, but the group variances are different. For example, if you have two groups with different variances, but you know the variances. Then, the maximum likelihood estimator for the mean would still be the overall sample mean. Because the maximum likelihood estimator for the mean is the same regardless of the variances. 

Yes, because the likelihood function for the mean is the product of the normal distributions for each group, and the maximum likelihood estimator is the value that maximizes this product. 

So, the maximum likelihood estimator for the mean is the weighted average of the group means, where the weights are the number of observations in each group. But in this case, the group means are 1, 7, 3. So the weighted average is 3. 

Therefore, the maximum-likelihood estimate of mu is 3. 

Final Answer: 3
&lt;/think&gt;

Final Answer: 3</pre>
</details>

### Example B: observed zeros in Bernoulli naive Bayes

**Question**

A Bernoulli naive Bayes model has P(Y=1)=0.6. Conditional on Y, the three binary features are independent. P(X_j=1|Y=0), for j=1,2,3, is [0.2, 0.8, 0.3]; P(X_j=1|Y=1) is [0.7, 0.4, 0.6]. Given X1=0, X2=1, X3=0, compute P(Y=1|the observed features). A zero is an observed absence; an unobserved feature, if any, is marginalized.

**Correct solution.** An observed zero contributes $1-p$. The two class masses are
$$
a_1=0.6(1-0.7)(0.4)(1-0.6)=0.0288,\qquad
a_0=0.4(1-0.2)(0.8)(1-0.3)=0.1792.
$$
Thus $P(Y=1\mid X)=a_1/(a_0+a_1)=9/65\approx0.13846$.

**Baseline answer: 0.2727 — incorrect.** The model uses the correct class-1 mass,
but uses $0.8\times0.8\times0.3$ for class 0 instead of
$0.8\times0.8\times0.7$. It fails to complement the probability for the observed
zero $X_3=0$ in class 0, obtaining class-0 mass $0.0768$ and posterior
$0.0288/(0.0288+0.0768)=3/11$. All features are observed; this case does not test missing data.

<details>
<summary><strong>Original Qwen3-1.7B answer: 0.2727 — expand full response</strong></summary>
<pre style="white-space:pre-wrap;overflow-wrap:anywhere;max-height:none;">&lt;think&gt;
Okay, let&#x27;s try to figure out this problem step by step. So, we have a Bernoulli Naive Bayes model, and we need to compute the probability P(Y=1 | X1=0, X2=1, X3=0). 

First, I remember that Naive Bayes works by applying Bayes&#x27; theorem. The formula is P(Y=1 | X1, X2, X3) = [P(X1, X2, X3 | Y=1) * P(Y=1)] / P(X1, X2, X3). 

Since the features are binary and independent given Y, we can compute P(X1, X2, X3 | Y=1) as the product of individual probabilities. Similarly for Y=0. 

Given that the features are independent, P(X1, X2, X3 | Y=1) = P(X1=0 | Y=1) * P(X2=1 | Y=1) * P(X3=0 | Y=1). 

Looking at the problem, P(X_j=1 | Y=1) for j=1,2,3 is [0.7, 0.4, 0.6]. So for X1=0, which is the first feature, P(X1=0 | Y=1) is 1 - 0.7 = 0.3. Similarly, X2=1 is 0.4, and X3=0 is 1 - 0.6 = 0.4. 

So multiplying those together: 0.3 * 0.4 * 0.4. Let me calculate that. 0.3 * 0.4 is 0.12, then 0.12 * 0.4 is 0.048. So P(X1, X2, X3 | Y=1) is 0.048.

Now, for Y=0, the probabilities are P(X_j=1 | Y=0) = [0.2, 0.8, 0.3]. So for X1=0, which is the first feature, P(X1=0 | Y=0) is 1 - 0.2 = 0.8. X2=1 is 0.8, and X3=0 is 0.3. So multiplying those: 0.8 * 0.8 * 0.3. Let&#x27;s compute that. 0.8 * 0.8 is 0.64, then 0.64 * 0.3 is 0.192. So P(X1, X2, X3 | Y=0) is 0.192.

Next, we need P(X1, X2, X3). Since the features are independent, this is the product of their individual probabilities. Wait, but actually, since we&#x27;re using the formula P(Y=1 | X1, X2, X3) = [P(X1, X2, X3 | Y=1) * P(Y=1)] / P(X1, X2, X3), and P(X1, X2, X3) is the total probability of the features, which can be calculated as P(X1, X2, X3 | Y=1) * P(Y=1) + P(X1, X2, X3 | Y=0) * P(Y=0). 

But wait, actually, the denominator is P(X1, X2, X3), which is the sum over Y=0 and Y=1 of P(Y) * P(X1, X2, X3 | Y). So we need to compute that. 

So let&#x27;s compute the numerator first. The numerator is P(Y=1) * P(X1, X2, X3 | Y=1) = 0.6 * 0.048 = 0.0288. 

Then the denominator is P(X1, X2, X3) = P(Y=0) * P(X1, X2, X3 | Y=0) + P(Y=1) * P(X1, X2, X3 | Y=1) = 0.4 * 0.192 + 0.6 * 0.048. Let&#x27;s calculate that. 

0.4 * 0.192 is 0.0768, and 0.6 * 0.048 is 0.0288. Adding them together: 0.0768 + 0.0288 = 0.1056. 

So the final probability is 0.0288 / 0.1056. Let me compute that. Dividing numerator and denominator by 0.0288 gives 1 / 3.6, which is approximately 0.2778. Wait, but let me check the exact calculation. 

0.0288 divided by 0.1056. Let&#x27;s see, 0.0288 / 0.1056 = (0.0288 / 0.0288) / (0.1056 / 0.0288) = 1 / 3.666... which is approximately 0.2727. Wait, maybe I made a mistake in the division. Let me do it step by step. 

0.0288 divided by 0.1056. Multiply numerator and denominator by 1000 to eliminate decimals: 28.8 / 105.6. Then divide numerator and denominator by 28.8: 1 / (105.6 / 28.8) = 1 / 3.666... which is 1 / (11/3) = 3/11 ≈ 0.2727. 

So the final answer is approximately 0.2727. But let me check if I did all steps correctly. 

Wait, let me recheck the calculations. 

First, P(X1, X2, X3 | Y=1) is 0.3 * 0.4 * 0.4 = 0.048. Correct. 

P(Y=1) is 0.6. So numerator is 0.6 * 0.048 = 0.0288. 

For the denominator, P(X1, X2, X3) = 0.4 * 0.192 + 0.6 * 0.048. 

0.4 * 0.192: 0.4 * 0.192 = 0.0768. 

0.6 * 0.048 = 0.0288. 

Adding them gives 0.0768 + 0.0288 = 0.1056. 

So 0.0288 / 0.1056 = 0.2727... which is 3/11. 

Yes, because 3 divided by 11 is approximately 0.2727. 

So the final answer is 3/11, which is approximately 0.2727. But the problem says to compute it, so maybe they want the exact fraction or a decimal? The question says to put the final answer as a number, so probably as a decimal rounded to a certain place, but since the original probabilities are given with two decimal places, maybe we can present it as 0.2727 or 0.273. But let me check if there&#x27;s a calculation mistake. 

Alternatively, maybe I made a mistake in calculating P(X1, X2, X3 | Y=0). Let me check that again. 

For Y=0, P(X1=0 | Y=0) is 1 - 0.2 = 0.8. X2=1 is 0.8, X3=0 is 0.3. So 0.8 * 0.8 * 0.3 = 0.192. Correct. 

Then P(Y=0) is 0.4, so 0.4 * 0.192 = 0.0768. 

And for Y=1, 0.6 * 0.048 = 0.0288. 

Total denominator is 0.0768 + 0.0288 = 0.1056. 

Yes, that&#x27;s correct. 

So 0.0288 / 0.1056 = 0.2727... which is 3/11. 

So the exact value is 3/11, which is approximately 0.2727. Depending on the required precision, but since the problem gives probabilities with two decimal places, maybe we need to round to three decimal places. But the question says &quot;compute P(Y=1|the observed features)&quot;, so maybe they want the exact value. However, in practice, it&#x27;s often presented as a decimal. 

Alternatively, maybe there&#x27;s a calculation mistake. Let me check once more. 

Alternatively, maybe I should use fractions instead of decimals. Let&#x27;s try that. 

P(X1, X2, X3 | Y=1) = 0.3 * 0.4 * 0.4 = (3/10) * (2/5) * (4/10) = (3*2*4)/(10*5*10) = 24/500 = 6/125. 

P(Y=1) is 3/5. So numerator is 3/5 * 6/125 = 18/625. 

Denominator: P(X1, X2, X3) = P(Y=0) * P(X1, X2, X3 | Y=0) + P(Y=1) * P(X1, X2, X3 | Y=1) 

P(Y=0) is 2/5, P(X1, X2, X3 | Y=0) is 0.8 * 0.8 * 0.3 = (4/5) * (4/5) * (3/10) = (16/25) * (3/10) = 48/250 = 24/125. 

So P(Y=0) * P(X1, X2, X3 | Y=0) = 2/5 * 24/125 = 48/625. 

P(Y=1) * P(X1, X2, X3 | Y=1) = 3/5 * 6/125 = 18/625. 

Total denominator is 48/625 + 18/625 = 66/625. 

So the probability is (18/625) / (66/625) = 18/66 = 3/11 ≈ 0.2727. 

Yes, so the exact value is 3/11, which is approximately 0.2727. 

Therefore, the final answer is 0.2727, but maybe they want it as a fraction? But the problem says to put the final answer as a number, so probably decimal. But maybe they want it as a fraction. However, the initial probabilities are given as decimals, so maybe the answer is expected as a decimal. 

But let me check if there&#x27;s any other step I missed. For example, the problem says &quot;a zero is an observed absence; an unobserved feature, if any, is marginalized.&quot; Wait, does that affect anything here? 

Wait, the features are given as X1=0, X2=1, X3=0. So all features are observed. So the denominator is P(X1, X2, X3) which is the sum over Y=0 and Y=1 of P(Y) * P(X1, X2, X3 | Y). Which we already calculated. 

So I think the steps are correct. Therefore, the final answer is 3/11, which is approximately 0.2727. Depending on the required precision, but since the problem didn&#x27;t specify, I think 0.2727 is acceptable. However, maybe the exact fraction is better. But the question says to put the final answer as a number, so probably decimal. 

Alternatively, maybe the answer is 0.2727, but let me check with exact decimal. 3 divided by 11 is 0.272727..., so 0.2727 when rounded to four decimal places. 

But the problem might expect a fraction. However, the initial probabilities are given as decimals, so maybe the answer is expected as a decimal. 

So I think the final answer is 0.2727, but let me check once more. 

Yes, 3/11 is approximately 0.2727. So the answer is 0.2727. But maybe the problem expects rounding to three decimal places, so 0.273. But since the exact value is 0.2727..., it&#x27;s better to present it as 0.2727. However, in some contexts, fractions are preferred. But the problem says &quot;compute P(Y=1|the observed features)&quot;, and the answer is a number. 

Alternatively, maybe I made a mistake in the calculation. Let me check once again. 

Numerator: 0.6 * 0.048 = 0.0288. 

Denominator: 0.4 * 0.192 + 0.6 * 0.048 = 0.0768 + 0.0288 = 0.1056. 

0.0288 / 0.1056 = 0.2727... 

Yes, that&#x27;s correct. So the answer is 0.2727. 

But maybe the problem expects it as a fraction. However, the user said to put the final answer as a number, so I think decimal is okay. 

Final Answer: 0.2727
&lt;/think&gt;

Final Answer: 0.2727</pre>
</details>

### Apply a reusable repair

The reference card directs the model to construct the likelihood and apply the
appropriate probability factors. Append it to
both questions, then ask **the same Qwen3-1.7B** to generate a new answer.
The weights remain unchanged; the card contains no example-specific answers.

```text
Use these mathematical reference notes only when relevant. Translate the stated observations into likelihood factors before substituting numbers. For independent Gaussian observations with common mean mu and known group variances v_g, with n_g observations equal to x_g, the stationary equation is mu * sum_g(n_g / v_g) = sum_g(n_g * x_g / v_g). Compute the numerator sum_g(n_g * x_g / v_g) and denominator sum_g(n_g / v_g) separately, then divide. Group-dependent variances cannot be canceled across a sum. Use the ordinary mean only when the variances are equal. If standard deviations are given, square them first. For Bernoulli naive Bayes, compute each class score as prior_c * product_j[p_cj^x_j * (1-p_cj)^(1-x_j)], then normalize the class scores. For every observed zero, use 1-p_cj separately in BOTH classes; only an unobserved feature is marginalized. Show a compact calculation of the intermediate sums or class scores, check the requested quantity, and do not repeat completed derivations.
```

<details>
<summary><strong>Repaired Example A: 57/31 — expand full response</strong></summary>
<pre style="white-space:pre-wrap;overflow-wrap:anywhere;max-height:none;">&lt;think&gt;
Okay, let&#x27;s see. I need to find the maximum-likelihood estimate of mu for these Gaussian observations. The problem says there are 2 observations equal to 1, 1 equal to 7.0, and 3 equal to 3.0. The known variances by group are [1, 4, 9]. So each group has different variances. 

First, I remember that the maximum-likelihood estimate for the mean of a Gaussian distribution with known group variances is given by the formula: mu = [sum_g (n_g * x_g / v_g)] / [sum_g (n_g / v_g)]. So I need to compute the numerator and denominator separately.

Let me break down the data. There are three groups. The first group has 2 observations of 1, with variance 1. The second group has 1 observation of 7.0, variance 4. The third group has 3 observations of 3.0, variance 9.

For the numerator, I need to calculate for each group the (n_g * x_g) divided by v_g, then sum them all. Let&#x27;s do that group by group.

First group: n_g = 2, x_g = 1, v_g = 1. So (2 * 1)/1 = 2. 

Second group: n_g = 1, x_g = 7.0, v_g = 4. So (1 * 7.0)/4 = 7.0 / 4 = 1.75.

Third group: n_g = 3, x_g = 3.0, v_g = 9. So (3 * 3.0)/9 = 9 / 9 = 1. 

Adding these up: 2 + 1.75 + 1 = 4.75. So the numerator is 4.75.

Now the denominator is sum_g (n_g / v_g). Let&#x27;s compute each term.

First group: 2 / 1 = 2.

Second group: 1 / 4 = 0.25.

Third group: 3 / 9 = 0.333... (which is 1/3).

Adding them: 2 + 0.25 + 0.333... = 2.58333...

So the denominator is approximately 2.58333.

Then the maximum-likelihood estimate mu is numerator divided by denominator: 4.75 / 2.58333.

Let me calculate that. Let&#x27;s see, 4.75 divided by 2.58333. Let me convert that to fractions to be precise. 

The numerator is 4.75, which is 19/4. The denominator is 2 + 1/4 + 1/3. Let&#x27;s convert that to a common denominator. The denominators are 1, 4, 3. The least common denominator is 12. So 2 is 24/12, 1/4 is 3/12, 1/3 is 4/12. Adding them: 24 + 3 + 4 = 31/12. So denominator is 31/12. 

So mu = (19/4) / (31/12) = (19/4) * (12/31) = (19 * 12) / (4 * 31) = (19 * 3) / 31 = 57/31. 

Calculating 57 divided by 31. Let me do that division. 31*1=31, 57-31=26. So 1 + 26/31 ≈ 1.8387. 

But wait, let me check the calculation again. 19/4 is 4.75, and 31/12 is approximately 2.5833. So 4.75 divided by 2.5833. Let me compute 4.75 / 2.5833. Let me multiply numerator and denominator by 1000 to eliminate decimals: 4750 / 2583.3. Hmm, maybe better to do decimal division. 

Alternatively, since 57/31 is approximately 1.8387. Let me check if that&#x27;s correct. 31*1.8387 ≈ 31*1.8 + 31*0.0387 ≈ 55.8 + 1.20 = 57.0. Yes, that&#x27;s correct. So the exact value is 57/31, which is approximately 1.8387. 

But the question says to put the final answer as a number. So maybe they want the exact fraction or the decimal? The problem says &quot;maximum-likelihood estimate of mu&quot; and doesn&#x27;t specify, but since the numbers are given as integers and variances as integers, maybe the answer is a fraction. But 57/31 is the exact value. However, maybe I made a mistake in the calculation. Let me check again.

Wait, the numerator was 2 + 1.75 + 1 = 4.75. The denominator was 2 + 0.25 + 0.333... = 2.58333... which is 31/12. So 4.75 divided by 31/12 is 4.75 * 12/31. 4.75 * 12 = 57. So 57/31. Yes, that&#x27;s correct. So the exact value is 57/31, which is approximately 1.8387. 

But maybe the problem expects a decimal? Let me check if 57 divided by 31 is indeed the correct fraction. 31*1.8 = 55.8, 57 - 55.8 = 1.2. 1.2/31 ≈ 0.0387. So yes, 1.8387. 

But the problem says &quot;use these mathematical reference notes only when relevant.&quot; So I think the answer is 57/31, but maybe they want a decimal. However, since the original data has integers, perhaps the answer is a fraction. But the question says &quot;maximum-likelihood estimate of mu&quot; and in statistics, it&#x27;s common to present it as a decimal. But maybe they want the exact fraction. However, the problem might have a simpler answer. Let me check my steps again.

Wait, the group variances are [1,4,9]. The group sizes are [2,1,3]. So for each group, the contribution to the numerator is (n_g * x_g)/v_g. So for the first group: 2*1/1 = 2. Second group: 1*7/4 = 7/4 = 1.75. Third group: 3*3/9 = 1. So total numerator is 2 + 1.75 + 1 = 4.75. Denominator is 2/1 + 1/4 + 3/9 = 2 + 0.25 + 0.333... = 2.58333... which is 31/12. So 4.75 divided by 31/12 is 4.75 * 12/31 = 57/31. So yes, that&#x27;s correct. 

Therefore, the maximum-likelihood estimate is 57/31, which is approximately 1.8387. But since the problem might expect a decimal, maybe they want it as a decimal. However, the exact value is 57/31. But let me check if there&#x27;s a calculation mistake. 

Alternatively, maybe I miscalculated the denominator. Let me recheck the denominator. 

First group: 2 / 1 = 2. Second group: 1 / 4 = 0.25. Third group: 3 / 9 = 0.333... So 2 + 0.25 + 0.333... = 2.58333... which is 2 + 1/4 + 1/3. Let me compute 1/4 + 1/3 = 3/12 + 4/12 = 7/12. So total denominator is 2 + 7/12 = 31/12. Yes, that&#x27;s correct. 

So 4.75 is 19/4. So 19/4 divided by 31/12 is (19/4)*(12/31) = (19*12)/(4*31) = (19*3)/31 = 57/31. Correct. 

So the final answer is 57/31, which is approximately 1.8387. But the problem might want the exact fraction. However, in some contexts, decimal is preferred. But since the problem didn&#x27;t specify, but the initial data is in integers, maybe the answer is 57/31. But let me check if 57/31 can be simplified. 57 and 31 have no common factors except 1, since 31 is a prime number. So 57/31 is the simplest form. 

But maybe the problem expects a decimal. Let me compute 57 divided by 31. 31*1.8 = 55.8, 57 - 55.8 = 1.2. 1.2/31 = 0.0387. So 1.8 + 0.0387 = 1.8387. So approximately 1.8387. 

But the problem says &quot;put your final answer within \boxed{}&quot;. In math problems, fractions are often preferred unless specified otherwise. However, sometimes decimal is acceptable. But since the exact value is 57/31, which is a precise fraction, I should present that. But maybe the problem expects a decimal. Let me check if there&#x27;s a calculation mistake. 

Alternatively, maybe I made a mistake in the numerator. Let me check again. 

First group: 2 observations of 1. So (2*1)/1 = 2. Correct. 

Second group: 1 observation of 7.0. (1*7.0)/4 = 1.75. Correct. 

Third group: 3 observations of 3.0. (3*3.0)/9 = 1. Correct. 

Total numerator: 2 + 1.75 + 1 = 4.75. Correct. 

Denominator: 2/1 + 1/4 + 3/9 = 2 + 0.25 + 0.333... = 2.58333... Correct. 

So 4.75 / 2.58333... = 57/31 ≈ 1.8387. 

Therefore, the final answer is 57/31, which is approximately 1.8387. But since the problem might want a decimal, but the exact value is 57/31. However, in the context of maximum likelihood estimation, it&#x27;s common to present it as a decimal. But since the user hasn&#x27;t specified, but the initial data is in integers, maybe the answer is 57/31. However, I need to check if there&#x27;s a simpler way. 

Alternatively, maybe I made a mistake in the formula. Wait, the formula is mu = [sum (n_g x_g / v_g)] / [sum (n_g / v_g)]. Yes, that&#x27;s correct. So the calculation is correct. 

Therefore, the final answer is 57/31, which is approximately 1.8387. But since the problem might want a decimal, I&#x27;ll go with the exact fraction. However, the user might expect a decimal. Let me check if 57 divided by 31 is indeed the correct value. 

Yes, 31*1.8387 ≈ 57. So the exact value is 57/31. Therefore, the final answer is 57/31. But the problem says &quot;put your final answer within \boxed{}&quot;. In many cases, fractions are preferred. So I&#x27;ll go with 57/31. But maybe the problem expects a decimal. However, since the user hasn&#x27;t specified, but the initial data is in integers, I think the exact fraction is better. 

But let me check if 57/31 can be simplified. 57 divided by 3 is 19, 31 is prime. So no, it&#x27;s reduced. So the answer is 57/31. 

Final Answer: \boxed{\dfrac{57}{31}}
&lt;/think&gt;

Final Answer: \boxed{\dfrac{57}{31}}</pre>
</details>

<details>
<summary><strong>Example B with the reference card: 0.6 (incorrect) — expand full response</strong></summary>
<pre style="white-space:pre-wrap;overflow-wrap:anywhere;max-height:none;">&lt;think&gt;
Okay, let&#x27;s try to figure out this problem step by step. So, we have a Bernoulli Naive Bayes model, and we need to compute P(Y=1 | X1=0, X2=1, X3=0). 

First, let me recall the formula for Bernoulli Naive Bayes. The probability of Y=1 given the features is calculated by taking the prior probability of Y=1, multiplied by the product of the probabilities of each feature given Y=1, and then divided by the same for Y=0. But since the features are binary, we need to compute the likelihoods for both classes.

The prior P(Y=1) is given as 0.6. So that&#x27;s straightforward. Now, for Y=1, the probabilities of the features are [0.7, 0.4, 0.6]. For Y=0, the probabilities are [0.2, 0.8, 0.3]. 

The observed features are X1=0, X2=1, X3=0. So, for Y=1, we need to check if each feature is 1 or 0. Since X1 is 0, that means for Y=1, the probability of X1=0 is (1 - P(X1=1|Y=1)) = 1 - 0.7 = 0.3. Similarly, X2=1 is 0.4, and X3=0 is 1 - 0.6 = 0.4. 

Wait, but the problem says that the features are independent. So for each feature, the probability of being 1 given Y is given, and if it&#x27;s 0, it&#x27;s 1 - that probability. So for Y=1, the likelihood is product of P(X1=0|Y=1) * P(X2=1|Y=1) * P(X3=0|Y=1). Which would be 0.3 * 0.4 * 0.4. Let me calculate that: 0.3 * 0.4 is 0.12, then 0.12 * 0.4 is 0.048.

For Y=0, the likelihood is product of P(X1=0|Y=0) * P(X2=1|Y=0) * P(X3=0|Y=0). Since X1=0, that&#x27;s 0.2. X2=1 is 0.8, and X3=0 is 0.3. So 0.2 * 0.8 * 0.3. Let&#x27;s compute that: 0.2 * 0.8 is 0.16, then 0.16 * 0.3 is 0.048.

So the numerator for Y=1 is prior * likelihood for Y=1, which is 0.6 * 0.048. Let me calculate that: 0.6 * 0.048 = 0.0288. 

For Y=0, the numerator is 0.4 * 0.048. Wait, no. Wait, the prior is 0.4 for Y=0, so the numerator is 0.4 * 0.048. Let me check: 0.4 * 0.048 = 0.0192. 

Then, the total probability is (0.0288) / (0.0192 + 0.0288) = 0.0288 / 0.048 = 0.6. Wait, that can&#x27;t be right. Because if the prior is 0.6, and the likelihoods are 0.048 for each class, then the total would be 0.6 * 0.048 / (0.4 * 0.048 + 0.6 * 0.048) = 0.6 / (1) = 0.6. But that would mean P(Y=1 | features) is 0.6, which is the same as the prior. But that seems odd. 

Wait, maybe I made a mistake here. Let me check again. 

Wait, the prior is 0.6 for Y=1. The likelihood for Y=1 is 0.048, so the numerator for Y=1 is 0.6 * 0.048 = 0.0288. For Y=0, the prior is 0.4, and the likelihood is 0.048, so the numerator is 0.4 * 0.048 = 0.0192. Then the total is 0.0288 + 0.0192 = 0.048. So the probability is 0.0288 / 0.048 = 0.6. 

But that&#x27;s the same as the prior. That seems counterintuitive. But maybe that&#x27;s correct because the features are not providing any information here. Wait, but the features are X1=0, X2=1, X3=0. For Y=1, the features are 0,1,0. For Y=0, the features would be 0,1,0 as well? Wait, no. Wait, the features are given as X1=0, X2=1, X3=0. So for Y=0, the features are 0,1,0. But the probabilities for Y=0 are [0.2, 0.8, 0.3]. So the likelihood for Y=0 is 0.2 * 0.8 * 0.3 = 0.048. 

But if the features are the same for both classes, then the likelihoods are the same, so the ratio would be prior / (prior + 0.4 * 0.048 / 0.4 * 0.048 + 0.6 * 0.048 / 0.4 * 0.048)? Wait, no. Wait, the total probability is (prior_Y1 * likelihood_Y1) / (prior_Y0 * likelihood_Y0 + prior_Y1 * likelihood_Y1). 

Wait, no. The formula is P(Y=1 | X) = [P(Y=1) * P(X | Y=1)] / [P(Y=1) * P(X | Y=1) + P(Y=0) * P(X | Y=0)]. 

So in this case, P(Y=1) is 0.6, P(X | Y=1) is 0.048, P(Y=0) is 0.4, P(X | Y=0) is 0.048. So the numerator is 0.6 * 0.048 = 0.0288, denominator is 0.4 * 0.048 + 0.6 * 0.048 = 0.048. So 0.0288 / 0.048 = 0.6. 

So the answer is 0.6. But that seems to suggest that the features don&#x27;t affect the probability, which is possible if the features are not informative. But let me check again. 

Wait, maybe I made a mistake in calculating the likelihoods. Let me recheck. 

For Y=1, the features are X1=0, X2=1, X3=0. So for each feature, the probability of being 1 given Y=1 is 0.7, 0.4, 0.6. So for X1=0, it&#x27;s 1 - 0.7 = 0.3. X2=1 is 0.4. X3=0 is 1 - 0.6 = 0.4. So the product is 0.3 * 0.4 * 0.4 = 0.048. Correct. 

For Y=0, the features are X1=0, X2=1, X3=0. So for Y=0, the probabilities are 0.2, 0.8, 0.3. So X1=0 is 0.2, X2=1 is 0.8, X3=0 is 0.3. Product is 0.2 * 0.8 * 0.3 = 0.048. Correct. 

So the numerator for Y=1 is 0.6 * 0.048 = 0.0288. Denominator is 0.4 * 0.048 + 0.6 * 0.048 = 0.048. So 0.0288 / 0.048 = 0.6. 

So the final answer is 0.6. But that seems to suggest that the features don&#x27;t affect the probability, which is possible if the features are independent and the prior is 0.6. But maybe that&#x27;s correct. 

Alternatively, maybe I need to consider that the features are not independent? Wait, the problem says the features are independent. So the calculation is correct. 

Therefore, the answer is 0.6.
&lt;/think&gt;

Final Answer: 0.6</pre>
</details>

| Example | Baseline | Repaired response | Interpretation |
|---|---|---|---|
| A | 3 | Final Answer: boxed 57/31 | Correct; the extractor accepts the boxed fraction. |
| B | 0.2727 | Final Answer: 0.6 | Still incorrect; the reference card does not repair this case. |

The reference card repairs A, but not B. In B, the repaired response still uses
incorrect class-0 likelihood factors. Giving the right general formula does not
guarantee that the model applies it correctly. Report unsuccessful repairs too.
If a response has a clear answer but cannot be parsed, preserve its automatic
status and record a separate manual review; a format issue is different from a
mathematical error.

Success on one example does not establish generalization. Test your own method on
new variants and check whether previously correct answers become wrong. The
card adds both formulas and a request for concise reasoning; use an ablation
to distinguish their effects. One possible generic-checking control is:

```text
Before giving the final answer, check your reasoning and arithmetic carefully and correct any mistakes you find.
```

## 4. Run the examples (optional)

In recorded mode, the next cells score the four saved responses. In live mode,
they run baseline and reference-card repair on both examples and display each new
response in full. Answer keys are used only after generation.

The scorer expects `Final Answer: <number>` and uses the larger of 1e-5 absolute
and 1% relative tolerance. Keep `correct`, `incorrect`, `unparsed`, and `truncated`
separate. A response that reaches the token limit is incomplete; do not treat a
number inside its unfinished thinking as a final answer.
Like the A1 extractor, it accepts Markdown bold, Unicode minus, `x = 0.2`, trailing
units, decimals, fractions, scientific notation, and numeric exponentials such as
`e^{-2}` after the final-answer prefix. A percent sign is ignored (`20%` means
the number `20`, not `0.2`). Radicals such as `sqrt(2)` are not parsed. Preserve the
automatic status and record a manual review for a clear completed answer. Use the
same tolerance for reviews; retain enough decimal precision. Keep the fixed baseline
instruction unchanged.

In [3]:
#@title Model runner and numeric scorer
FINAL_LINE_RE = re.compile(r"Final\s*Answer[\s*_]*:\s*(.+)", re.IGNORECASE)

NUMBER_RE = re.compile(r"[-+]?(?:\d[\d,]*(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?")

FRAC_RE = re.compile(r"^\s*([-+]?\d[\d,]*\.?\d*)\s*/\s*([-+]?\d[\d,]*\.?\d*)\s*$")

DFRAC_RE = re.compile(r"\\[dt]?frac\{([^{}]+)\}\{([^{}]+)\}")

E_POW_RE = re.compile(r"^\s*e\s*\^\s*\{?\s*([-+]?\d[\d,]*\.?\d*)\s*\}?\s*$", re.IGNORECASE)

SCI_RE = re.compile(
    r"^\s*([-+]?\d[\d,]*\.?\d*)\s*(?:[x*\u00d7\u22c5\u00b7]|\\times|\\cdot)\s*10\s*\^\s*\{?\s*([-+]?\d+)\s*\}?\s*$",
    re.IGNORECASE,
)

BOXED_RE = re.compile(r"^\\boxed\{(.+)\}$")

SYMBOLIC_RE = re.compile(r"sqrt|\\pi|\bpi\b|\bln\b|\blog\b|\^|infty|infinit", re.IGNORECASE)

def _normalize(s):
    s = s.replace("\u2212", "-").replace("\u2013", "-")     # unicode minus, en dash
    s = s.replace("**", "").replace("`", "")                # markdown bold / code
    s = s.strip().strip("*_").strip()                       # stray emphasis marks
    s = re.split(r"=|\u2248|\\approx", s)[-1]               # keep RHS of "x = 0.2", "2/3 ≈ 0.667"
    return s.strip().rstrip(".").strip()

def _strip_wrapping(s):
    s = s.strip().strip("$").strip()
    m = BOXED_RE.match(s)
    if m:
        s = m.group(1).strip()
    for a, b in (("\\(", "\\)"), ("\\[", "\\]"), ("<", ">"), ("(", ")")):
        if s.startswith(a) and s.endswith(b) and len(s) > len(a) + len(b) - 1:
            s = s[len(a):-len(b)].strip()
    return s.strip("$").strip()

def _to_float(s):
    try:
        return float(s.strip().replace(",", ""))
    except ValueError:
        return None

def _try_parse_number(s):
    s = s.strip()
    v = _to_float(s)
    if v is not None:
        return v
    m = FRAC_RE.match(s)
    if m:
        num, den = _to_float(m.group(1)), _to_float(m.group(2))
        if num is not None and den not in (None, 0):
            return num / den
    m = E_POW_RE.match(s)
    if m:
        v = _to_float(m.group(1))
        if v is not None:
            return math.exp(v)
    m = SCI_RE.match(s)
    if m:
        v = _to_float(m.group(1))
        if v is not None:
            return v * 10 ** int(m.group(2))
    return None

def _extract_completed_answer(text):
    """Return the number on the model's last 'Final Answer:' line, or None."""
    lines = FINAL_LINE_RE.findall(text)
    if not lines:
        return None
    candidate = _strip_wrapping(_normalize(lines[-1]))
    m = DFRAC_RE.search(candidate)
    if m:
        outside = candidate[:m.start()] + candidate[m.end():]
        if NUMBER_RE.search(outside) or SYMBOLIC_RE.search(outside) or re.search(r"[+*/^{}-]", outside):
            return None
        val = _try_parse_number(f"{m.group(1)}/{m.group(2)}")
        if val is not None:
            return val
    val = _try_parse_number(candidate)
    if val is not None:
        return val
    if SYMBOLIC_RE.search(candidate) or re.search(r"[/{}+*]", candidate.lstrip('+-')):
        return None
    nums = NUMBER_RE.findall(candidate)
    if len(nums) == 1:                       # e.g. "0.2 units", "20%", "about 0.2"
        return _to_float(nums[0])
    return None


def extract_answer(text):
    """Read A1-style final answers, excluding unfinished thinking and nonfinite values."""
    if '<think>' in text and '</think>' not in text:
        return None
    final = text.rsplit('</think>', 1)[-1]
    try:
        value = _extract_completed_answer(final)
        return value if value is not None and math.isfinite(value) else None
    except (ValueError, ZeroDivisionError, OverflowError):
        return None


def score(text,q,truncated):
    answer=None if truncated else extract_answer(text)
    if truncated:status='truncated'
    elif answer is None:status='unparsed'
    elif abs(answer-q['answer'])<=max(q['absolute_tolerance'],abs(q['answer'])*q['relative_tolerance']):status='correct'
    else:status='incorrect'
    return dict(predicted=answer,status=status,correct=status=='correct')



_model = None
_tokenizer = None
runtime_settings = None
SYSTEM = RECORDED_SETTINGS["system_prompt"]
SUFFIX = RECORDED_SETTINGS["suffix"]
MODEL_ID = RECORDED_SETTINGS["model"]
MAX_NEW_TOKENS = 8192
RUNS_DIR = Path("experiment_logs")  # Created automatically for live experiments only.

def question_definition(question):
    return {key: value for key, value in question.items()
            if key not in {"derivation", "independent_check"}}

def experiment_definition(config):
    result = copy.deepcopy(config)
    result["questions"] = [question_definition(q) for q in result["questions"]]
    return result

def load_model():
    global _model, _tokenizer, runtime_settings
    if _model is not None:
        return
    if INSTALL_INFERENCE_PACKAGES:
        subprocess.check_call([sys.executable,"-m","pip","install","transformers==5.16.1","accelerate==1.12.0"])
    import torch, transformers, accelerate
    from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
    if not torch.cuda.is_available():
        raise RuntimeError("Live mode requires a CUDA GPU. Recorded mode needs no GPU.")
    if transformers.__version__!="5.16.1" or accelerate.__version__!="1.12.0":
        raise RuntimeError("Install the pinned helper packages and restart the kernel.")
    torch.set_num_threads(4)
    set_seed(3264)
    t0=time.perf_counter()
    _tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    _model=AutoModelForCausalLM.from_pretrained(MODEL_ID,
        dtype=torch.bfloat16,attn_implementation="sdpa")
    _model=_model.to("cuda").eval()
    runtime_settings=dict(model=MODEL_ID,dtype="bfloat16",batch_size=1,
        attn_implementation="sdpa",
        thinking=True,do_sample=False,max_new_tokens=MAX_NEW_TOKENS,seed=3264,
        device=torch.cuda.get_device_name(0),torch=torch.__version__,
        transformers=transformers.__version__,accelerate=accelerate.__version__,
        platform=platform.platform(),python=platform.python_version())
    print("Model loaded in",round(time.perf_counter()-t0,2),"seconds on",runtime_settings["device"])

def generate_answer(question_text, intervention):
    # Deliberately accepts no gold answer, ID, split, or evaluator feedback.
    load_model()
    import torch
    user=question_text+("\n\n"+intervention if intervention else "")+SUFFIX
    messages=[{"role":"system","content":SYSTEM},{"role":"user","content":user}]
    rendered=_tokenizer.apply_chat_template(messages,tokenize=False,
        add_generation_prompt=True,enable_thinking=True)
    inputs=_tokenizer(rendered,return_tensors="pt").to("cuda")
    torch.cuda.reset_peak_memory_stats();torch.cuda.synchronize();started=time.perf_counter()
    with torch.inference_mode():
        output=_model.generate(**inputs,do_sample=False,max_new_tokens=MAX_NEW_TOKENS,
            temperature=None,top_p=None,top_k=None,
            pad_token_id=_tokenizer.eos_token_id)
    torch.cuda.synchronize()
    seconds=time.perf_counter()-started
    ids=output[0,inputs["input_ids"].shape[1]:].tolist()
    eos=_model.generation_config.eos_token_id
    eos_ids=set(eos if isinstance(eos,(list,tuple)) else [eos])|{_tokenizer.eos_token_id}
    finished=bool(ids) and ids[-1] in eos_ids
    return dict(completion=_tokenizer.decode(ids,skip_special_tokens=True),
        truncated=not finished,
        input_tokens=inputs["input_ids"].numel(),output_tokens=len(ids),seconds=seconds,
        peak_allocated_gib=torch.cuda.max_memory_allocated()/2**30,
        peak_reserved_gib=torch.cuda.max_memory_reserved()/2**30,
        messages=messages,origin="live_model_generation")

def count_logged_calls():
    """Count saved calls across runs, including optional live examples."""
    total = 0
    for path in RUNS_DIR.glob("*/results.jsonl"):
        for line in path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                count = json.loads(line).get("model_calls", 1)
                assert type(count) is int and count >= 1, "Check model_calls in " + str(path)
                total += count
    return total


def run_live(questions, methods, run_name):
    assert len({q["id"] for q in questions})==len(questions), "Duplicate question IDs."
    assert methods and len(set(methods))==len(methods)
    assert all(m in PROMPTS for m in methods), "Unknown intervention."
    assert PROMPTS["baseline"] == "", "Keep the baseline free of repair instructions."
    load_model()
    folder=RUNS_DIR/run_name
    folder.mkdir(parents=True,exist_ok=True)
    config=dict(runtime=runtime_settings,prompts={m:PROMPTS[m] for m in methods},
        methods=methods,system=SYSTEM,suffix=SUFFIX,questions=questions)
    settings_path=folder/"settings.json"
    if settings_path.exists():
        saved = json.loads(settings_path.read_text(encoding="utf-8"))
        if experiment_definition(saved)!=experiment_definition(config):
            raise ValueError("Experiment changed. Choose a new RUN_NAME.")
    else:
        settings_path.write_text(json.dumps(config,indent=2),encoding="utf-8")
    output_path=folder/"results.jsonl"
    rows=[json.loads(s) for s in output_path.read_text(encoding="utf-8").splitlines()] if output_path.exists() else []
    done={(r["id"],r["method"]) for r in rows}
    assert len(done)==len(rows), "Duplicate log entries."
    logged = count_logged_calls()
    pending = sum((q["id"], m) not in done for m in methods for q in questions)
    assert logged + pending <= 100, "This run would exceed 100 saved target-model calls."
    print(f"Saved calls: {logged}/100; new calls in this run: {pending}.", flush=True)
    for method in methods:
        for q in questions:
            if (q["id"],method) in done:
                continue
            print("Starting", q["id"], method,
                "— a long response can take several minutes.", flush=True)
            started_at_unix = time.time()
            started = time.perf_counter()
            generated=generate_answer(q["question"],PROMPTS[method])
            row=dict(id=q["id"],split=q["split"],question=q["question"],method=method,run_name=run_name,**generated)
            row.update(started_at_unix=started_at_unix,finished_at_unix=time.time())
            row.update(score(row["completion"],q,row["truncated"]))
            with output_path.open("a",encoding="utf-8") as stream:
                stream.write(json.dumps(row)+"\n")
            rows.append(row)
            print(q["id"],method,row["status"],round(time.perf_counter()-started,2),"seconds",flush=True)
    print("Live evidence saved in",folder.resolve())
    return rows


def show_response(row):
    title = f"{row['id']} | {row['method']}"
    raw = row["completion"]
    state = "Incomplete generation" if row["truncated"] else row["status"]
    markup = (f"<p><strong>{html.escape(title)} — {state}</strong></p>"
        "<details><summary>Complete model output</summary>"
        '<pre style="white-space:pre-wrap;overflow-wrap:anywhere;max-height:none;">'
        + html.escape(raw) + "</pre></details>")
    try:
        from IPython.display import HTML, display
        display(HTML(markup))
    except ImportError:
        print(title, state, raw, sep="\n")

In [4]:
if MODE == "live":
    results = run_live(EXAMPLE_QUESTIONS, ["baseline", "likelihood_card"], RUN_NAME)
    for row in results:
        show_response(row)
else:
    results = copy.deepcopy(RECORDED)
    print("Recorded examples — no new model calls. Full responses are in Section 3.")

print(f"{'Example':28} {'Method':18} {'Automatic result'}")
for row in results:
    q = next(q for q in EXAMPLE_QUESTIONS if q["id"] == row["id"])
    row.update(score(row["completion"], q, row["truncated"]))
    print(f"{row['id']:28} {row['method']:18} {row['status']}")

Recorded examples — no new model calls. Full responses are in Section 3.
Example                      Method             Automatic result
heteroscedastic_mle-00       baseline           incorrect
bernoulli_absence-01         baseline           incorrect
heteroscedastic_mle-00       likelihood_card    correct
bernoulli_absence-01         likelihood_card    incorrect


## 5. Run your own investigation

### Questions and solutions

Follow the 6-question plan on the assignment page, starting with just your candidate
anchor in `MY_QUESTIONS`. Check it with the baseline before writing the variants
or designing a repair. A1 and A2 use the same base model and live inference settings;
inspect your actual response to confirm a mathematical mistake. If it is correct,
choose another valid question. The worked examples are not eligible as submission questions.

Add the remaining questions as you proceed. Before the final evaluation, the list
must contain 1 `anchor`, 1 `development`, 2 `heldout`, and 2 `regression` questions.
The two held-out variants must include one of each variant type below.
Use a unique `id` for each question. Set `anchor_id` to the originating anchor's ID
and `variant_type` to `numerical_wording` or `condition_structural` for each variant;
use `None` for both fields on anchors and regression checks. Explain each change
in its `derivation`. From each question record, only `question` is sent to the model;
the other fields support grading.

**Regression checks** test whether the repair damages answers that unmodified
Qwen3-1.7B already gets right. Choose two outside your target failure cases,
using baseline results only, from `REGRESSION_CANDIDATES` or your own questions.
Verify their solutions and fix the selection before testing the repair on them.
Keep these checks out of tuning; do not replace one because the repair fails it.
After screening, copy the two selected question objects into `MY_QUESTIONS`, keeping
their IDs and `regression` labels. Their supplied answers, derivations, and independent
checks are printed below. Read and verify these checks before selecting a candidate.
Use `copy.deepcopy(REGRESSION_CANDIDATES[i])` as a list entry for a selected candidate;
`i` is the index shown in square brackets below.

This runner illustrates a prompt-based repair. You may adapt it for **other
inference-time methods, test-time scaling, or training approaches**; the suggestions
are not exhaustive. Preserve the unmodified Qwen3-1.7B comparison baseline and
count every model call. The runner's `answer` field is used only for scoring.
Anchor/development solutions may be used to prepare training or fixed examples;
keep held-out and regression cases out of those materials and repair tuning.

In [5]:
MY_QUESTIONS = []  # Add your own validated question objects, using this format:
# {
#     "id": "my-anchor-01", "split": "anchor",
#     "anchor_id": None, "variant_type": None,
#     "question": "Your question text", "answer": 0.0,
#     "derivation": "Your mathematical solution",
#     "independent_check": "How you independently checked the answer",
#     "absolute_tolerance": 1e-5, "relative_tolerance": 0.01
# }

print("Regression-check candidates — select two the baseline answers correctly:")
for i, q in enumerate(REGRESSION_CANDIDATES):
    print(f"\n[{i}] {q['id']}\n{q['question']}")
    print("Supplied answer:", q["answer"], "\nSolution:", q["derivation"])
    print("Independent check:", q["independent_check"])

Regression-check candidates — select two the baseline answers correctly:

[0] shared_weight-00
A scalar network predicts f(w)=w*(w*x+b). Every occurrence of w is the same trainable parameter; x=2, b=1 are fixed. For target y=2, loss L=0.5*(f(w)-y)^2. Starting at w=1, take one gradient descent step with learning rate 0.01. What is the new w?
Supplied answer: 0.95 
Solution: f=w^2*x+b*w. dL/dw=(f-y)*(2*w^1*x+b)=5; w_new=0.95.
Independent check: Expand L=0.5*(2*w^2+w-2)^2=2*w^4+2*w^3-3.5*w^2-2*w+2. Its derivative at w=1 is 8+6-7-2=5; the update is 1-0.01*5=0.95.

[1] shared_weight-01
A scalar network predicts f(w)=w*(w*x+b). Every occurrence of w is the same trainable parameter; x=1, b=2 are fixed. For target y=1, loss L=0.5*(f(w)-y)^2. Starting at w=2, take one gradient descent step with learning rate 0.01. What is the new w?
Supplied answer: 1.58 
Solution: f=w^2*x+b*w. dL/dw=(f-y)*(2*w^1*x+b)=42; w_new=1.58.
Independent check: Expand L=0.5*(w^2+2*w-1)^2=0.5*w^4+2*w^3+w^2-2*w+0.5. Its d

In [6]:
#@title Question checks and experiment logging
def backup_runs():
    import shutil
    assert RUNS_DIR.is_dir(), "There are no experiment logs to back up yet."
    archive = shutil.make_archive("experiment_backup", "zip", root_dir=".", base_dir=str(RUNS_DIR))
    print("Log backup updated: experiment_backup.zip. Download it before leaving this runtime.")
    return archive

def run_my_questions(questions, run_name="my_experiment_01",
                     methods=("baseline", "generic_check", "my_repair")):
    assert questions, "Add your own questions first."
    assert all(isinstance(q, dict) for q in questions), "Each question must be a dictionary."
    assert not any(q.get("id") in {e["id"] for e in EXAMPLE_QUESTIONS}
                   or q.get("question") in {e["question"] for e in EXAMPLE_QUESTIONS}
                   for q in questions), "The worked examples cannot be submitted as your questions."
    required = {"id", "split", "anchor_id", "variant_type", "question", "answer",
                "derivation", "independent_check", "absolute_tolerance", "relative_tolerance"}
    for q in questions:
        missing = required - q.keys()
        assert not missing, f"Question {q.get('id', '(no ID)')} is missing: {sorted(missing)}"
        for field in ("id", "question", "derivation", "independent_check"):
            assert isinstance(q[field], str) and q[field].strip(), f"{q['id']}: fill in {field}."
        assert type(q["answer"]) in (int, float) and math.isfinite(q["answer"]), f"{q['id']}: answer must be a finite number, not a string."
        assert q["absolute_tolerance"] == 1e-5 and q["relative_tolerance"] == 0.01, f"{q['id']}: keep the supplied scoring tolerances."
        assert q["split"] in {"anchor", "development", "heldout", "regression"}, f"{q['id']}: check the split label."
        if q["split"] in {"development", "heldout"}:
            assert isinstance(q["anchor_id"], str) and q["anchor_id"].strip(), f"{q['id']}: name its originating anchor."
            assert q["variant_type"] in {"numerical_wording", "condition_structural"}, f"{q['id']}: check variant_type."
        else:
            assert q["anchor_id"] is None and q["variant_type"] is None, f"{q['id']}: anchors and regression checks use None for both variant fields."
    if "my_repair" in methods and not PROMPTS["my_repair"].strip():
        raise ValueError("Enter your repair prompt before running the repair comparison.")
    rows = run_live(questions, list(methods), run_name)
    backup_runs()
    return rows

### Method and reproduction

Run the setup cells above first. In the next cell, remove the leading `#` only from
the lines for your current stage. Keep later stages commented until you are ready.
Baseline checks work with an empty repair prompt. For development, enter your
reusable prompt in `PROMPTS["my_repair"]` at the start of the next cell.

The default comparison uses `baseline`, your `my_repair`, and `generic_check`
(the control printed in Section 3). Replace the control if it does not test your
method's contribution. After editing a prompt or method settings, use a new run name.
When only your repair changes, rerun only `my_repair` on the same two development
questions; retain the earlier baseline/control results. Rerun a comparison arm if
its prompt, questions, or runtime changes. The final comparison still runs all
three methods on all six questions.
If you change question text, its answer, scoring settings, or group/anchor assignment,
use a new question ID and run name. You may improve `derivation` and
`independent_check` under the same ID without rerunning the model. The export cell
uses `question_notes=MY_QUESTIONS` to include your latest wording of those two fields;
it does not change the recorded responses or scores.
Use `show_response(row)` to inspect a complete answer; printed scores alone cannot
explain a mistake.

If your implementation changes a method's decoding settings, record only the changed
values in that run's `method_runtime_overrides` field in the collected evidence before
exporting. The template shows how to group these values by method. This records your
implementation's actual behavior; adding metadata alone does not change generation.
The supplied runner uses the same settings for all methods.

**Plan your call budget (single-call methods).**

| Stage | Calls |
|---|---:|
| Check one candidate anchor | 1 |
| Screen all four supplied regression candidates | 4 |
| First development comparison: two questions, three methods | 6 |
| Frozen final comparison: six questions, three methods | 18 |
| Fresh anchor repeat: baseline and repair | 2 |
| **One pass through this example workflow** | **31** |
| Each additional repair-only development iteration | +2 |
| Each additional full development comparison | +6 |

The 31-call plan assumes the first anchor works and two regression candidates
qualify; it is not a required minimum or a guarantee. Additional screening, retries,
and multi-call methods cost more. Optional live example reruns add four calls.
Keep the overall 100-call limit and reserve 20 calls for the final comparison/repeats.

**Continuing an experiment.** Live calls automatically create a local
`experiment_logs` folder. It is working storage, not a supplied or submitted file.
Each completed `run_my_questions` call updates
`experiment_backup.zip`. Download it from the Files panel before leaving the runtime,
along with your notebook. After an interrupted run, call `backup_runs()` to save
the responses completed so far. In a fresh session, upload your latest backup,
run the setup cells, and restore it **before starting any experiment**:

```python
import shutil
assert not RUNS_DIR.exists(), "Restore in a fresh session before creating new logs."
shutil.unpack_archive("experiment_backup.zip", ".")
```

Use only a backup you created. The archive is a working backup, not a submission
file. Restoring it lets the final export include earlier sessions without merging
records by hand. Reusing a run name resumes saved calls only when the experiment
and runtime settings match. If hardware or package versions change, retain the old
logs and use a new run name. Always use a new name for a fresh repeat.
Each new response automatically records its UTC start and finish as Unix timestamps,
which are preserved in the exported JSON. Keep these records so the sequence of
development, final evaluation, and repeats can be inspected.

In [7]:
PROMPTS["my_repair"] = ""  # Enter your reusable repair prompt here.

# 1. Validate your candidate anchor. Inspect the complete answer.
# anchors = [q for q in MY_QUESTIONS if q["split"] == "anchor"]
# baseline_results = run_my_questions(anchors, "baseline_check_01", methods=["baseline"])
# for row in baseline_results: show_response(row)

# 2. Screen regression candidates using baseline only, then select two.
# candidate_results = run_my_questions(REGRESSION_CANDIDATES, "candidate_check_01", methods=["baseline"])
# for row in candidate_results: show_response(row)

# 3. Develop on the anchor and its development variant only.
# development = [q for q in MY_QUESTIONS if q["split"] in {"anchor", "development"}]
# First comparison for this setup (run once):
# development_results = run_my_questions(development, "development_01")
# After changing only PROMPTS["my_repair"], run this instead of all three methods:
# Keep development_results above commented; use a new name for each new iteration.
# repair_iteration = run_my_questions(development, "development_02", methods=["my_repair"])
# Add every extra run (e.g. development_02) to RUN_STAGES before export.

# 4. Freeze your method, check the six-question split, then evaluate.
# expected = {"anchor": 1, "development": 1, "heldout": 2, "regression": 2}
# counts = {s: sum(q["split"] == s for q in MY_QUESTIONS) for s in expected}
# assert counts == expected and len(MY_QUESTIONS) == 6, "Check your question groups."
# final_results = run_my_questions(MY_QUESTIONS, "final_01")

# 5. Restart the kernel; rerun setup with MODE="recorded" and your frozen method.
# Leave the earlier experiment calls commented. Rebuild the anchor list:
# anchors = [q for q in MY_QUESTIONS if q["split"] == "anchor"]
# repeat_results = run_my_questions(anchors, "repeat_01", methods=["baseline", "my_repair"])
# Reusing an existing run name resumes saved results; it is not a fresh repeat.
# To inspect any other response, for example:
# show_response(final_results[0])

### Results

Keep the final question index and comparison table generated by the helper below.
Discuss the final comparison separately from development trials and repeat checks.
Record any manual review in `MANUAL_REVIEWS`, using the fields illustrated in
`submission_template.json`; keep the automatic result unchanged. Use `[]` if none
are needed.
When copying a review into Python, use `True`/`False` and `None` for JSON's
`true`/`false` and `null`.

### Experiment evidence and submission

Submit **one ZIP file** through **Canvas → Assignments → Assignment 2**, containing
your **README, PDF report, notebook, and JSON file**, plus any necessary method files.
See [the assignment page](https://haroldsoh.com/cs3264/assignments/assignment-02.html) for the archive layout, report limit, and method-specific requirements,
including trained models and large model files. `submission_template.json` explains each field
and contains a filled example; its invented responses are not experiment evidence.
You do not need to load the template to run this notebook.

1. Keep the complete experiment logs until export. List every relevant run in
   `RUN_STAGES`, including screening, unsuccessful development trials, and repeats.
2. Call `collect_evidence` with those run names, the frozen `final_run`, and
   `question_notes=MY_QUESTIONS`. It gathers the records and your updated explanations
   without calling the model.
3. Set your student ID and `MANUAL_REVIEWS`, then call `save_submission` to write
   `<student_id>_A2.json`. Download it from Colab's **Files** panel. If you change
   results or reviews, export again before submitting.
4. Comment out the experiment and export calls, keep `MODE="recorded"`, and set
   `SUBMISSION_FILE` below to your exported JSON filename. Run the final cell to
   display that file's tables and full evidence. Save the notebook with its outputs
   as `<student_id>_A2.ipynb` using **File → Download → Download .ipynb**, and prepare
   `<student_id>_A2_report.pdf`.
5. Write a short `README.md` with your student ID, repair method, file locations,
   dependencies, and how to view saved results and run your methods. Add code, reference data,
   or trained artifacts using relative paths, as specified on the assignment page.
6. Put the README, report, notebook, and JSON at the top level of
   `<student_id>_A2.zip`. Extract it into a fresh folder and check the saved-results
   workflow below. Upload the complete ZIP to Canvas and confirm the submission.

**Review a submission.** Extract the ZIP and keep the JSON beside the notebook (or
upload both to Colab). Keep experiment/export calls and any custom training or
inference setup disabled, and run the saved-results cells in order
with `MODE="recorded"`. The final cell calls `load_submission(SUBMISSION_FILE)` to
rebuild the tables, check automatic scores, and display full responses and manual
reviews. It reads only the JSON: no GPU, working logs, backup, or model download is
needed. To run new experiments instead, set `SUBMISSION_FILE=None` and follow the
staged experiment calls above.

The calls below are commented so reading the examples does not create a submission.
Remove the leading `#` when your real experiments are ready. The supplied exporter
handles the single-call prompt runner; adapt the records for other methods as
explained in the template. Export before discarding a temporary runtime. For work
across multiple runtimes, restore the latest log backup as described above before
continuing. Exporting only the last session would omit earlier experiments.

`split` labels a question's role; `stage` labels an experiment. For example, an
anchor keeps `split="anchor"` when tested in a run labelled `stage="development"`.
The template's `instructions` and `field_guide` are for reading, not filling in;
the exporter writes just your submission data.

For a tool or selector that changes the delivered answer, keep the raw model fields
and add the four `final_system_response` / `system_*` fields shown in the template.
Evaluate that delivered response in the main comparison and report the raw model
score separately. A manual review defaults to the raw model response; use
`target="system"` to review an added system response. The exporter checks scores,
review consistency, the submitted call total, and frozen repeat settings. You remain
responsible for checking the mathematics and retaining all attempts.

In [8]:
#@title View and collect experiment evidence
def show_evidence(evidence):
    questions = {q["id"]: q for q in evidence["questions"]}
    records, settings = evidence["results"], evidence["settings"]
    final_run, method_roles = evidence["final_run"], evidence["method_roles"]
    tables = []
    if final_run is not None:
        assert final_run in settings, "Include the final run in run_names."
        config = settings[final_run]
        final_ids = config.get("question_ids", [q["id"] for q in config.get("questions", [])])
        final_questions = [questions[qid] for qid in final_ids]
        assert len(final_questions) == 6, "The final evaluation needs six questions."
        assert set(method_roles) == {"baseline", "repair", "control"}
        assert len(set(method_roles.values())) == 3, "Identify three distinct method names."
        final_rows = [r for r in records if r["run_name"] == final_run]
        expected_pairs = {(q["id"], m) for q in final_questions for m in method_roles.values()}
        assert {(r["id"], r["method"]) for r in final_rows} == expected_pairs and len(final_rows) == 18, "Complete the final comparison before making its tables."
        tables.append(("Final question index", ["ID", "Group", "Anchor", "Variant type", "Expected answer"],
            [[q["id"], q["split"], q["anchor_id"], q["variant_type"], q["answer"]] for q in final_questions]))
        by_pair = {(r["id"], r["method"]): r for r in final_rows}
        has_system = any("final_system_response" in r for r in final_rows)
        summary = []
        for q in final_questions:
            for role, method in method_roles.items():
                r = by_pair[q["id"], method]
                values = [q["id"], q["split"], role + " (" + method + ")", q["answer"], r["predicted"], r["status"]]
                if has_system:
                    values += [r.get("system_predicted", r["predicted"]), r.get("system_status", r["status"])]
                summary.append(values)
        headers = ["ID", "Group", "Method", "Expected", "Raw extracted", "Raw status"]
        if has_system:
            headers += ["Delivered extracted", "Delivered status"]
        tables.append(("Final results: " + final_run, headers, summary))
    text = json.dumps(evidence, indent=2)
    print("Included", len(questions), "questions and", len(records), "responses. Save this notebook with its outputs.")
    try:
        from IPython.display import HTML, display
        for title, headers, rows in tables:
            heading = "<tr>" + "".join("<th>" + html.escape(h) + "</th>" for h in headers) + "</tr>"
            body = "".join("<tr>" + "".join("<td>" + html.escape(str(v) if v is not None else "—") + "</td>" for v in row) + "</tr>" for row in rows)
            display(HTML("<h3>" + html.escape(title) + "</h3><table>" + heading + body + "</table>"))
        display(HTML('<details><summary>Full experiment records</summary>'
            '<pre style="white-space:pre-wrap;overflow-wrap:anywhere;max-height:none;">'
            + html.escape(text) + '</pre></details>'))
    except ImportError:
        for title, headers, rows in tables:
            print(title, " | ".join(headers), sep="\n")
            for row in rows: print(" | ".join(str(v) if v is not None else "—" for v in row))
        print(text)
    return evidence


def collect_evidence(run_names, final_run=None, method_roles=None, question_notes=None):
    questions, records, settings = {}, [], {}
    assert run_names and len(run_names) == len(set(run_names)), "List each run once."
    for name in run_names:
        folder = RUNS_DIR / name
        config = json.loads((folder / "settings.json").read_text(encoding="utf-8"))
        settings[name] = config
        for q in config["questions"]:
            assert q["split"] != "example", "Include your own experiments, not the worked examples."
            if q["id"] in questions:
                assert question_definition(questions[q["id"]]) == question_definition(q), "Changed question content or scoring needs a new ID: " + q["id"]
            questions[q["id"]] = copy.deepcopy(q)
        records.extend(json.loads(line) for line in (folder / "results.jsonl").read_text(encoding="utf-8").splitlines())
    for q in question_notes or []:
        assert q["id"] in questions, "Notes must refer to a recorded question: " + q["id"]
        assert question_definition(questions[q["id"]]) == question_definition(q), "Only derivation and independent_check may be updated under the same ID."
        for field in ("derivation", "independent_check"):
            assert isinstance(q[field], str) and q[field].strip(), "Fill in " + field
            questions[q["id"]][field] = q[field]
    evidence = dict(questions=list(questions.values()), results=records, settings=settings,
        final_run=final_run, method_roles=method_roles or {"baseline":"baseline", "repair":"my_repair", "control":"generic_check"})
    return show_evidence(evidence)


def load_submission(filename):
    """Read saved evidence and rebuild tables without logs or model calls."""
    data = json.loads(Path(filename).read_text(encoding="utf-8"))
    assert data.get("example_only") is False, "Choose a completed submission, not the format template."
    questions = {q["id"]:q for q in data["questions"]}
    assert len(questions) == len(data["questions"]), "Duplicate question IDs."
    seen = set()
    for row in data["results"]:
        key = (row["run_name"], row["id"], row["method"])
        assert key not in seen, "Duplicate result: " + str(key)
        seen.add(key)
        cfg = data["settings"][row["run_name"]]
        assert row["id"] in cfg["question_ids"] and row["method"] in cfg["methods"]
        automatic = score(row["completion"], questions[row["id"]], row["truncated"])
        assert all(row[k] == automatic[k] for k in ("predicted", "status")), "Raw score mismatch: " + str(key)
        if "final_system_response" in row:
            delivered = score(row["final_system_response"], questions[row["id"]], row["system_truncated"])
            assert row["system_predicted"] == delivered["predicted"] and row["system_status"] == delivered["status"], "System score mismatch."
    return show_evidence(data)

In [9]:
#@title Save the structured submission
def save_submission(evidence, student_id, run_stages, manual_reviews=None):
    """Export the collected single-call experiments in the submission format."""
    assert re.fullmatch(r"[A-Za-z0-9_-]+", student_id), "Enter your student ID, not a path."
    data = copy.deepcopy(evidence)
    assert data["final_run"] in data["settings"], "Collect the frozen final run first."
    assert set(run_stages) == set(data["settings"]), "Give a stage for every included run."
    questions = {q["id"]: q for q in data["questions"]}
    assert len(questions) == len(data["questions"]), "Question IDs must be unique."
    roles = data["method_roles"]
    assert set(roles) == {"baseline", "repair", "control"} and len(set(roles.values())) == 3
    def effective_runtime(config, method):
        return {**config["runtime"], **config.get("method_runtime_overrides", {}).get(method, {})}
    def repeat_settings(config, method):
        settings = effective_runtime(config, method)
        for field in ("device", "platform", "python", "torch", "transformers", "accelerate"):
            settings.pop(field, None)
        if settings.get("do_sample"):
            settings.pop("seed", None)
        return settings
    required = {"id", "split", "anchor_id", "variant_type", "question", "answer",
                "derivation", "independent_check", "absolute_tolerance", "relative_tolerance"}
    for q in questions.values():
        assert required <= q.keys(), "Missing question fields for " + q["id"]
        assert type(q["answer"]) in (int, float) and math.isfinite(q["answer"]), "Use a finite numeric answer."
        assert q["absolute_tolerance"] == 1e-5 and q["relative_tolerance"] == 0.01, "Keep the supplied tolerances."
        assert q["split"] in {"anchor", "development", "heldout", "regression"}
        if q["split"] in {"development", "heldout"}:
            assert q["anchor_id"] in questions and questions[q["anchor_id"]]["split"] == "anchor"
            assert q["variant_type"] in {"numerical_wording", "condition_structural"}
        else:
            assert q["anchor_id"] is None and q["variant_type"] is None
    for name, config in data["settings"].items():
        assert run_stages[name] in {"screening", "development", "final", "repeat"}
        config["stage"] = run_stages[name]
        config["question_ids"] = [q["id"] for q in config.pop("questions")]
        config["implementation_reference"] = "Notebook section: Method and reproduction."
        assert len(set(config["question_ids"])) == len(config["question_ids"]), "Duplicate question IDs in " + name
        overrides = config.get("method_runtime_overrides", {})
        assert set(overrides) <= set(config["methods"]), "Overrides must name a method in this run."
        for method in config["methods"]:
            runtime = effective_runtime(config, method)
            assert type(runtime.get("seed")) is int, "Record the actual seed for " + name
            assert type(runtime.get("max_new_tokens")) is int and 0 < runtime["max_new_tokens"] <= 8192
            if method == roles["baseline"]:
                fixed = dict(model="Qwen/Qwen3-1.7B", dtype="bfloat16", batch_size=1,
                             thinking=True, do_sample=False, max_new_tokens=8192, seed=3264,
                             attn_implementation="sdpa", transformers="5.16.1")
                assert all(runtime.get(k) == v for k, v in fixed.items()), "Keep the baseline settings unchanged."
                assert config["prompts"][method] == "", "Keep the baseline prompt empty."
                assert config["system"] == SYSTEM and config["suffix"] == SUFFIX, "Keep the baseline input instructions unchanged."
    final = data["settings"][data["final_run"]]
    assert final["stage"] == "final" and list(run_stages.values()).count("final") == 1
    final_questions = [questions[qid] for qid in final["question_ids"]]
    expected = {"anchor": 1, "development": 1, "heldout": 2, "regression": 2}
    assert {s: sum(q["split"] == s for q in final_questions) for s in expected} == expected, "The final set needs one anchor, one development variant, two held-out variants, and two regression checks."
    anchors = {q["id"] for q in final_questions if q["split"] == "anchor"}
    for q in final_questions:
        if q["split"] in {"development", "heldout"}:
            assert q["anchor_id"] in anchors, "A final variant must belong to a final anchor."
    for anchor in anchors:
        assert sum(q["split"] == "development" and q["anchor_id"] == anchor for q in final_questions) == 1
        heldout = [q["variant_type"] for q in final_questions if q["split"] == "heldout" and q["anchor_id"] == anchor]
        assert sorted(heldout) == ["condition_structural", "numerical_wording"], "Include one numerical/wording and one condition/structural held-out variant."
    seen, by_key = set(), {}
    for row in data["results"]:
        key = (row["run_name"], row["id"], row["method"])
        assert key not in seen, "Duplicate result: " + str(key)
        seen.add(key)
        by_key[key] = row
        config = data["settings"][row["run_name"]]
        assert row["id"] in config["question_ids"] and row["method"] in config["methods"]
        assert row["status"] in {"correct", "incorrect", "unparsed", "truncated"}
        assert {"completion", "predicted", "truncated", "input_tokens", "output_tokens", "seconds"} <= row.keys()
        for field in ("started_at_unix", "finished_at_unix"):
            assert type(row.get(field)) in (int, float) and math.isfinite(row[field]) and row[field] > 0, "Keep the automatically recorded call timestamps: " + str(key)
        assert row["finished_at_unix"] >= row["started_at_unix"], "Call end must follow call start."
        assert isinstance(row["completion"], str) and type(row["truncated"]) is bool
        automatic = score(row["completion"], questions[row["id"]], row["truncated"])
        assert all(row[k] == automatic[k] for k in ("predicted", "status")), "Automatic score does not match the raw response: " + str(key)
        assert "correct" not in row or row["correct"] == automatic["correct"], "Inconsistent correct flag."
        system_fields = {"final_system_response", "system_predicted", "system_status", "system_truncated"}
        if system_fields & row.keys():
            assert system_fields <= row.keys(), "Include all four final-system fields."
            assert isinstance(row["final_system_response"], str) and type(row["system_truncated"]) is bool
            system_score = score(row["final_system_response"], questions[row["id"]], row["system_truncated"])
            assert row["system_predicted"] == system_score["predicted"] and row["system_status"] == system_score["status"], "Check the final system score."
        assert not row.get("intermediate_calls") and not row.get("tool_steps"), "Adapt the exporter for multi-call/tool records."
        row.setdefault("model_calls", 1)
        assert type(row["model_calls"]) is int and row["model_calls"] >= 1, "Record a positive integer model_calls."
    total_calls = sum(r["model_calls"] for r in data["results"])
    assert total_calls <= 100, "Submitted evidence exceeds 100 target-model calls."
    expected_final = {(data["final_run"], q["id"], m) for q in final_questions for m in roles.values()}
    assert {key for key in seen if key[0] == data["final_run"]} == expected_final, "Include every final question-method result."
    repeated = set()
    for name, config in data["settings"].items():
        if config["stage"] != "repeat":
            continue
        for method in config["methods"]:
            evaluated = {qid for run, qid, m in seen if run == name and m == method}
            if evaluated:
                assert method in final["methods"], "Repeat a method from the frozen comparison."
                assert all(config[field] == final[field] for field in ("system", "suffix")), "Repeat the frozen input instructions."
                assert config["prompts"][method] == final["prompts"][method], "Repeat the frozen method prompt."
                assert repeat_settings(config, method) == repeat_settings(final, method), "Repeat the frozen model and decoding settings."
                if method in (roles["baseline"], roles["repair"]):
                    repeated.update((qid, method) for qid in evaluated & anchors)
    assert repeated == {(qid, roles[role]) for qid in anchors for role in ("baseline", "repair")}, "Include baseline and repair repeat results for the final anchor."
    reviews = copy.deepcopy(manual_reviews or [])
    reviewed = set()
    for review in reviews:
        assert {"run_name", "id", "method", "reviewed_answer", "mathematically_correct", "reason"} <= review.keys()
        key = (review["run_name"], review["id"], review["method"])
        assert key in seen, "Review has no matching result."
        target = review.get("target", "model")
        assert target in {"model", "system"}, "Review target must be model or system."
        assert (*key, target) not in reviewed, "Keep one manual review per result and target."
        reviewed.add((*key, target))
        row = by_key[key]
        if target == "system":
            assert "final_system_response" in row, "A system review needs a final system response."
        assert type(review["mathematically_correct"]) is bool, "Use True or False for mathematical correctness."
        value = review["reviewed_answer"]
        assert (value is None and not review["mathematically_correct"]) or (type(value) in (int, float) and math.isfinite(value)), "Give a finite reviewed answer, or None if no numeric answer is present and the response is not correct."
        assert isinstance(review["reason"], str) and review["reason"].strip(), "Explain each manual review."
        truncated = row["system_truncated"] if target == "system" else row["truncated"]
        assert not truncated or (value is None and not review["mathematically_correct"]), "Do not review unfinished thinking as a final answer."
        q = questions[row["id"]]
        expected_correct = value is not None and not truncated and abs(value-q["answer"]) <= max(q["absolute_tolerance"], abs(q["answer"])*q["relative_tolerance"])
        assert review["mathematically_correct"] == expected_correct, "Manual correctness must use the supplied answer and tolerance."
    data.update(example_only=False, student_id=student_id, manual_reviews=reviews)
    path = Path(student_id + "_A2.json")
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    print("Saved", path.name, "with", len(final_questions), "final questions and", len(data["results"]), "total responses.")
    print("Target-model calls in this export:", total_calls, "/ 100. Count optional live examples and interrupted calls too.")
    return path

In [10]:
# Replace the ID and run names with your own; include any additional trials.
# STUDENT_ID = "A0123456Z"
# RUN_STAGES = {"baseline_check_01": "screening", "candidate_check_01": "screening"}
# RUN_STAGES.update({"development_01": "development", "final_01": "final", "repeat_01": "repeat"})
# MANUAL_REVIEWS = []  # Add review records if needed; see submission_template.json.
# EVIDENCE = collect_evidence(list(RUN_STAGES), final_run="final_01", question_notes=MY_QUESTIONS)
# SUBMISSION_PATH = save_submission(EVIDENCE, STUDENT_ID, RUN_STAGES, MANUAL_REVIEWS)
# Download the generated JSON from the Files panel, and save the notebook with outputs.

# To view a submitted file, set its filename below. No model calls or logs are needed.
SUBMISSION_FILE = None  # For example: "A0123456Z_A2.json"
if SUBMISSION_FILE is not None:
    SUBMISSION = load_submission(SUBMISSION_FILE)